# Covariate Stack — Germany, 100 m (EPSG:32632)
Builds the environmental covariate stack for the synthetic experiment (`synthetic_experiment.ipynb.nosync Kopie.ipynb`): download and align every layer to one 100 m UTM 32N grid, log1p-transform the right-skewed layers, analyse collinearity and select the final layers, and save the selected, **unscaled** stack. No species data, no splits, no scaling — the synthetic experiment scales the stack when it loads it.

| Step | Description |
|------|-------------|
| 1 | Setup & configuration |
| 2 | Administrative boundaries & study area |
| 3 | Elevation: Copernicus DEM GLO-30 → UTM 32N, 100 m (mean aggregation) |
| 4–5 | Pixel-size check, grid template |
| — | Terrain derivatives (slope, TRI, roughness) |
| 6a–6i | Covariates: winter temperature (BIO11), population, road and railway density, distance to water, habitat score (ESA WorldCover), protected areas, NDVI, summer precipitation (BIO18) |
| 8 | Stack assembly (14 layers) & resolution table |
| 9 | Figure of the raw stack |
| 10 | log1p transformation |
| 11 | Collinearity analysis & variable selection |
| 12–13 | Distributions and map of the final layers |
| 14 | Checkpoint for the synthetic experiment |


## 1 — Setup & Imports

In [ ]:
# PROJECT ROOT. Set the environment variable PROJECT_ROOT to the folder that
# holds code/ and data/, or edit the fallback below.
PROJECT_ROOT = os.environ.get("PROJECT_ROOT", os.path.abspath(".."))
# ============================================================
# STEP 1 — Imports & Setup
# ============================================================
# Before writing any analysis code we import all the external
# libraries (packages) that we will use later. Python does not
# load packages automatically — we must explicitly import them.

# ── PROJ database: pin it to THIS environment ─────────────────────────────────
# The base conda env ships an old `proj4` package whose activation script sets
# PROJ_LIB to base's proj.db, which belongs to a different PROJ version. Inside an
# activated wolf_sdm kernel that is harmless -- PROJ_DATA points to the right
# database and takes precedence -- but started any other way (a terminal with only
# base active, a batch script) every EPSG lookup and reprojection fails with
# "proj.db lacks DATABASE.LAYOUT.VERSION". Pointing both variables at the running
# interpreter's own database makes the notebook independent of how it was started.
# Must run BEFORE rasterio / pyproj / geopandas are imported.
import os, sys
_proj_dir = os.path.join(sys.prefix, "share", "proj")
if os.path.exists(os.path.join(_proj_dir, "proj.db")):
    os.environ["PROJ_DATA"] = _proj_dir
    os.environ["PROJ_LIB"]  = _proj_dir

import os           # operating system utilities: file paths, directory creation
import glob         # find files matching a pattern (e.g. all .shp files in a folder)
import numpy as np  # fast numerical arrays — the backbone of all raster operations
import pandas as pd # tabular data (DataFrames) — used for occurrence records & metrics
import geopandas as gpd  # extends pandas with geometry/spatial operations

import rasterio                            # read / write GeoTIFF raster files
from rasterio.transform import from_bounds # build an affine transform from corner coords
from rasterio.warp import (                # coordinate reprojection tools
    calculate_default_transform, reproject, Resampling
)
from rasterio.mask import mask as rio_mask         # clip a raster to polygon geometries
from rasterio.features import (
    geometry_mask,                                  # rasterize a mask from geometries
    rasterize as rio_rasterize                      # burn vector features into a raster
)
from rasterio.enums import MergeAlg                # controls how overlapping polygons are merged

from scipy.ndimage import (
    generic_filter,          # apply any function inside a moving window
    uniform_filter,          # fast local mean (uniform box filter)
    distance_transform_edt   # Euclidean distance to nearest non-zero pixel
)

import matplotlib.pyplot as plt           # standard Python plotting library
import matplotlib.colors as mcolors       # colour maps and normalisation
from matplotlib.gridspec import GridSpec  # flexible subplot layout
import seaborn as sns                     # statistical visualisation (heatmaps etc.)

from sklearn.metrics import roc_curve, auc  # ROC curve and AUC computation

from shapely.geometry import Point, box   # create geometric objects
from shapely.ops import unary_union       # merge many geometries into one

import requests   # download files from the internet
import zipfile    # unzip downloaded archives
import io         # in-memory byte streams (avoid writing temp files to disk)
import warnings
warnings.filterwarnings("ignore")  # suppress non-critical warnings to keep output clean

# Code outputs (figures, checkpoints, computed stacks) and gathered/downloaded
# data live in separate sibling folders, outside the synced "code" directory.
CODE_OUT = os.path.join(PROJECT_ROOT, "code.nosync")
DATA_DIR = os.path.join(PROJECT_ROOT, "data.nosync")
os.makedirs(os.path.join(CODE_OUT, "figures"), exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, "map_data"), exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, "layers"), exist_ok=True)
print("Imports OK.")

# ── Species-specific packages ────────────────────────────────────────────────
import pygbif.occurrences as occ  # query the GBIF occurrence database

# TensorFlow / Keras — deep learning framework used to build and train the CNN
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers  # building blocks for neural network layers

import matplotlib.patches as mpatches  # used to create custom legend items

# On Apple Silicon (M1/M2) TensorFlow would try to allocate all GPU memory at once
# and crash the kernel. memory_growth=True lets it grow the allocation gradually.
import gc  # Python garbage collector — used to free memory between CV folds
for _gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(_gpu, True)

print(f"TensorFlow: {tf.__version__}")


## 1a — Configuration
`TARGET_RES` sets the analysis resolution. Every cache file and the checkpoint folder carry it in their name, so changing it builds a separate set of outputs instead of overwriting the existing ones. `N_FOLDS`, `THIN_RES`, `INNER_DIST`, `OUTER_DIST` and `BLOCK_SIZE_M` belong to the real-data pipeline and are not used in this notebook.

In [ ]:
# ============================================================
# STEP 1a — Configuration
# ============================================================
SPECIES       = "sus_scrofa"   # used in all directory and file names
TARGET_RES    = 100      # metres — analysis grid resolution; tags every cache/checkpoint filename
PATCH         = 32       # CNN input patch size, in pixels (PATCH x PATCH)

FIG_DIR = os.path.join(CODE_OUT, "figures", SPECIES)
os.makedirs(FIG_DIR, exist_ok=True)


## 2 — Administrative Boundaries & Study Area
Download Germany GADM level-1 (Bundesland) boundaries. The study area covers the whole country.

In [ ]:
# ============================================================
# STEP 2 — Study Area (Administrative Boundaries)
# ============================================================

def download_gadm_germany(path=os.path.join(DATA_DIR, "map_data")):
    """Download and return Germany GADM level-1 (Bundesland) boundaries as a GeoDataFrame."""
    url = "https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_DEU_shp.zip"
    if not os.path.exists(os.path.join(path, "gadm41_DEU_1.shp")):
        print("Downloading GADM Germany boundaries...")
        r = requests.get(url)
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            z.extractall(path)
    return gpd.read_file(os.path.join(path, "gadm41_DEU_1.shp"))

germany = download_gadm_germany()
regions = germany.to_crs("EPSG:4326")

# 16 German federal states (Bundesländer) — used later to split the GBIF
# query (Step 14), since GBIF's search API caps pagination at 100k records
# per query and a single country-wide query for a common species like
# Sus scrofa would exceed that.
GERMAN_STATES = [
    "Baden-Württemberg", "Bayern", "Berlin", "Brandenburg", "Bremen",
    "Hamburg", "Hessen", "Mecklenburg-Vorpommern", "Niedersachsen",
    "Nordrhein-Westfalen", "Rheinland-Pfalz", "Saarland", "Sachsen",
    "Sachsen-Anhalt", "Schleswig-Holstein", "Thüringen",
]

print(f"Study area: {len(regions)} polygons (all 16 German federal states).")
regions[["NAME_1", "geometry"]].head(16)


In [ ]:
# Create the study area overview map — all 16 German federal states (Bundesländer).

import urllib.request, zipfile, io
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe

gadm_shp = f"{DATA_DIR}/map_data/gadm41_DEU_1.shp"
if not os.path.exists(gadm_shp):
    url = "https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_DEU_shp.zip"
    with urllib.request.urlopen(url) as r:
        with zipfile.ZipFile(io.BytesIO(r.read())) as z:
            z.extractall(os.path.join(DATA_DIR, "map_data"))

study_area = gpd.read_file(gadm_shp).to_crs("EPSG:4326")

# Sized for a SMALL placement in the thesis: every font is scaled up by
# roughly 1.7 so the map stays readable when the figure is reduced, and the
# in-figure title is gone because the caption carries it.
fig, ax = plt.subplots(figsize=(7, 9))
study_area.plot(ax=ax, color="#4a90d9", edgecolor="white", linewidth=0.8, alpha=0.85)

SHORT_NAMES = {
    "Nordrhein-Westfalen":    "Nordrhein-\nWestfalen",
    "Mecklenburg-Vorpommern": "Mecklenburg-\nVorpommern",
    "Schleswig-Holstein":     "Schleswig-\nHolstein",
    "Baden-Württemberg":      "Baden-\nWürttemberg",
    "Sachsen-Anhalt":         "Sachsen-\nAnhalt",
}
# Berlin is an enclave AT Brandenburg's centroid, so the two labels coincide.
# At 6.5 pt they merely touched; at 11 pt Berlin disappeared underneath. Only
# the label moves, never the geometry.
LABEL_OFFSETS = {"Brandenburg": (0.30, -0.62)}

for _, row in study_area.iterrows():
    cx, cy = row.geometry.centroid.x, row.geometry.centroid.y
    dx, dy = LABEL_OFFSETS.get(row["NAME_1"], (0.0, 0.0))
    cx, cy = cx + dx, cy + dy
    label  = SHORT_NAMES.get(row["NAME_1"], row["NAME_1"])
    ax.text(cx, cy, label, ha="center", va="center",
            fontsize=11, fontweight="bold", color="white",
            path_effects=[pe.withStroke(linewidth=2.2, foreground="#1a4a7a")])

study_area.boundary.plot(ax=ax, color="#1a4a7a", linewidth=0.8)

minx, miny, maxx, maxy = study_area.total_bounds
ax.set_xlim(minx - 0.8, maxx + 0.8)
ax.set_ylim(miny - 0.75, maxy + 0.8)

scale_x0, scale_y = minx + 0.25, miny - 0.2
scale_x1 = scale_x0 + 1.0
ax.plot([scale_x0, scale_x1], [scale_y, scale_y], color="black", lw=2)
ax.plot([scale_x0, scale_x0], [scale_y - 0.05, scale_y + 0.05], color="black", lw=2)
ax.plot([scale_x1, scale_x1], [scale_y - 0.05, scale_y + 0.05], color="black", lw=2)
ax.text((scale_x0 + scale_x1) / 2, scale_y - 0.34, "approx. 75 km",
        ha="center", fontsize=13)

ax.annotate("", xy=(scale_x0 - 0.40, scale_y + 0.10),
            xytext=(scale_x0 - 0.40, scale_y - 0.22),
            arrowprops=dict(arrowstyle="-|>", color="black", lw=1.8))
ax.text(scale_x0 - 0.40, scale_y + 0.12, "N",
        ha="center", va="bottom", fontsize=18, fontweight="bold")

legend_handles = [
    mpatches.Patch(color="#4a90d9", alpha=0.85, label="Study area (16 federal states)"),
]
ax.legend(handles=legend_handles, loc="upper left", fontsize=14, framealpha=0.9)

ax.set_xlabel("Longitude (\u00b0E)", fontsize=15)
ax.set_ylabel("Latitude (\u00b0N)", fontsize=15)
ax.tick_params(labelsize=13)

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/Study_Area_Map.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {CODE_OUT}/figures/Study_Area_Map.png")


## 3 — Elevation Data & Spatial Template
Download **Copernicus DEM GLO-30** (30 m native, public domain) tiles via HTTPS, reproject to **UTM Zone 32N** and aggregate to **100 m** in one pass with mean resampling (equivalent to R's `aggregate(..., fun='mean')`), then mask to Germany. This raster is the grid template every other layer is aligned to.

| Step | R equivalent |
|------|--------------|
| Download 30 m tiles, mosaic | `get_elev_raster(..., z=10)` |
| Reproject to UTM 32N and aggregate to 100 m (`Resampling.average`) | `project()` + `aggregate(..., fun='mean')` |
| Mask to Germany | `mask(elev, regions_sf)` |

In [ ]:
# iCloud Drive reports lower free space than actual disk — disable GDAL's overly conservative check.
import os
os.environ["GDAL_CHECK_DISK_FREE_SPACE"] = "FALSE"

from rasterio.merge import merge
import math

# Helper functions for downloading, reprojecting, and masking the elevation raster.
# All raster layers are ultimately aligned to this elevation template (100 m UTM 32N).

def download_copernicus_dem(gdf, path=os.path.join(DATA_DIR, "map_data")):
    """
    Download Copernicus DEM GLO-30 (30m, public domain) tiles via HTTPS.
    Uses rasterio vsicurl — no extra packages or login needed.
    """
    out_path = os.path.join(os.path.expanduser("~/wolf_sdm_stack"), "elevation_raw.tif")
    if os.path.exists(out_path):  # skip download if already done
        print(f"Elevation file already exists: {out_path}")
        return out_path

    minx, miny, maxx, maxy = gdf.total_bounds   # bounding box of the study area in WGS84 degrees
    # Compute the grid of 1°×1° DEM tiles that cover the bounding box
    lons = range(int(math.floor(minx)), int(math.ceil(maxx)))
    lats = range(int(math.floor(miny)), int(math.ceil(maxy)))

    base = "https://copernicus-dem-30m.s3.eu-central-1.amazonaws.com"
    datasets = []
    print(f"Opening Copernicus DEM tiles ({len(list(lons))} x {len(list(lats))} grid)...")
    for lat in lats:
        for lon in lons:
            # Tile naming convention: N/S + 2-digit lat, E/W + 3-digit lon
            ns = "N" if lat >= 0 else "S"
            ew = "E" if lon >= 0 else "W"
            name = (f"Copernicus_DSM_COG_10_{ns}{abs(lat):02d}_00_"
                    f"{ew}{abs(lon):03d}_00_DEM")
            # /vsicurl/ prefix tells rasterio to stream the file over HTTPS
            # without downloading the whole thing first
            url = f"/vsicurl/{base}/{name}/{name}.tif"
            try:
                ds = rasterio.open(url)
                datasets.append(ds)
            except Exception:
                pass  # tile doesn't exist for this lat/lon (e.g. sea)

    if not datasets:
        raise RuntimeError("No DEM tiles found — check internet connection.")

    # Merge all individual tiles into one continuous mosaic array
    print(f"Merging {len(datasets)} tiles...")
    mosaic, transform = merge(datasets)

    profile = datasets[0].profile.copy()
    profile.update({
        "height":    mosaic.shape[1],
        "width":     mosaic.shape[2],
        "transform": transform,
        "driver":    "GTiff",
        "dtype":     "float32",
        "compress":  "lzw",   # lossless compression to keep file size small
    })
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(mosaic.astype(np.float32))
    for ds in datasets:
        ds.close()
    print(f"Saved merged DEM → {out_path}")
    return out_path


def reproject_raster(src_path, dst_crs="EPSG:32632", res=None,
                     resampling=Resampling.bilinear, out_path=None):
    """
    Reproject a raster to a target CRS.
    res=None  → preserve native pixel size after reprojection.
    res=100   → force output to exactly 100 m pixels.
    Resampling.bilinear smooths values between pixels (good for continuous data like elevation).
    """
    with rasterio.open(src_path) as src:
        # calculate_default_transform works out the new grid dimensions and transform
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds,
            resolution=res   # None = preserve native resolution
        )
        kwargs = src.meta.copy()
        kwargs.update({"crs": dst_crs, "transform": transform,
                       "width": width, "height": height, "dtype": "float32",
                       "compress": "lzw", "bigtiff": "IF_NEEDED"})  # IF_NEEDED enables BigTIFF (>4 GB) automatically
        data = np.empty((1, height, width), dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=data[0],
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=dst_crs,
            resampling=resampling,
            dst_nodata=np.nan,   # pixels outside the source extent become NaN
        )
    if out_path is None:
        out_path = src_path.replace(".tif", f"_{dst_crs.replace(':', '')}.tif")
    with rasterio.open(out_path, "w", **kwargs) as dst:
        dst.write(data)
    return out_path


def aggregate_to_target_res(src_path, target_res=100, out_path=None):
    """
    Resample raster to target_res metres using average resampling.
    Resampling.average computes the mean of all source pixels that fall
    within each output pixel — equivalent to R's aggregate(..., fun='mean').
    """
    with rasterio.open(src_path) as src:
        current_res = src.transform.a   # pixel width in metres (x-direction)
        print(f"Current resolution: {current_res:.1f} m  →  target: {target_res} m "
              f"(factor ≈ {target_res/current_res:.2f})")

        transform, width, height = calculate_default_transform(
            src.crs, src.crs, src.width, src.height, *src.bounds,
            resolution=target_res
        )
        kwargs = src.meta.copy()
        kwargs.update({"transform": transform, "width": width,
                       "height": height, "dtype": "float32",
                       "compress": "lzw"})
        data = np.empty((1, height, width), dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=data[0],
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=src.crs,
            resampling=Resampling.average,   # mean aggregation, mirrors R aggregate()
            dst_nodata=np.nan,
        )
    if out_path is None:
        out_path = src_path.replace(".tif", f"_{target_res}m.tif")
    with rasterio.open(out_path, "w", **kwargs) as dst:
        dst.write(data)
    print(f"Aggregated raster saved → {out_path}")
    return out_path


def mask_raster_to_gdf(raster_path, gdf, out_path=None):
    """Clip a raster to the extent of a GeoDataFrame (e.g. study region polygons)."""
    gdf_proj = gdf.to_crs(rasterio.open(raster_path).crs)   # match raster CRS first
    geoms = [geom.__geo_interface__ for geom in gdf_proj.geometry]
    with rasterio.open(raster_path) as src:
        # crop=True trims the output extent to the polygon bounding box
        out_image, out_transform = rio_mask(src, geoms, crop=True, nodata=np.nan)
        out_meta = src.meta.copy()
    out_meta.update({"transform": out_transform, "width": out_image.shape[2],
                     "height": out_image.shape[1], "nodata": np.nan, "dtype": "float32"})
    if out_path is None:
        out_path = raster_path.replace(".tif", "_masked.tif")
    with rasterio.open(out_path, "w", **out_meta) as dst:
        dst.write(out_image.astype(np.float32))
    return out_path


# Pipeline — mirrors the R equivalent workflow:
# 1. Download 30 m Copernicus DEM tiles → mosaic
# 2. Reproject + aggregate directly to TARGET_RES UTM (avoids >4 GB intermediate)
# 3. Mask to study area boundaries

elev_masked = f"{DATA_DIR}/layers/elevation_masked_{TARGET_RES}m.tif"

if os.path.exists(elev_masked):
    print(f"Elevation already processed: {elev_masked}")
else:
    # Step 1: Download Copernicus DEM (30m, WGS84)
    elev_path_raw = download_copernicus_dem(regions)

    # Step 2: Reproject from WGS84 directly to UTM 32N at TARGET_RES.
    # Resampling.average averages all ~11 source pixels per output pixel —
    # equivalent to a separate aggregate(fun="mean") step, but in one pass.
    # Output is ~350 MB instead of the >4 GB 30m intermediate.
    _utm_100m_path = f"{DATA_DIR}/map_data/elevation_utm_{TARGET_RES}m.tif"
    if os.path.exists(_utm_100m_path):
        print(f"UTM {TARGET_RES}m elevation already exists: {_utm_100m_path}")
        elev_path_100m = _utm_100m_path
    else:
        elev_path_100m = reproject_raster(
            elev_path_raw, dst_crs="EPSG:32632", res=TARGET_RES,
            resampling=Resampling.average,
            out_path=_utm_100m_path
        )
    with rasterio.open(elev_path_100m) as src:
        print(f"Reprojected elevation: {src.transform.a:.1f} m resolution, "
              f"{src.width} x {src.height} px")

    # Step 3: Mask to the study area — pixels outside Germany become NaN
    elev_masked = mask_raster_to_gdf(
        elev_path_100m, regions,
        out_path=elev_masked
    )
    print(f"Done. Final elevation raster: {TARGET_RES} m UTM, masked to study area.")

## 4 — Pixel Dimension Analysis
Check the actual pixel size in metres at the centre of the study area.

In [ ]:
# ============================================================
# STEP 4 — Pixel Dimension Check
# ============================================================
# Before processing we verify the spatial resolution of the
# elevation raster. The affine transform stored inside a GeoTIFF
# describes how pixel indices map to real-world coordinates.
# transform.a = pixel width (x), transform.e = pixel height (y, negative for top-down rasters).

with rasterio.open(elev_masked) as src:
    res_x = abs(src.transform.a)   # pixel width  in the CRS units (metres if projected)
    res_y = abs(src.transform.e)   # pixel height in the CRS units
    crs   = src.crs

# is_projected: True if CRS uses metres (UTM etc.), False if it uses degrees (WGS84)
if crs.is_projected:
    print("--- Pixel Size Analysis ---")
    print(f"CRS: {crs.to_epsg()} (projected — units already in metres)")
    print(f"Resolution: {res_x:.1f} m × {res_y:.1f} m")
else:
    # Convert degrees to metres using approximate scaling factors at the study area centre
    center_lat = 51.0   # approximate central latitude of Germany
    pixel_width_m  = res_x * 111320 * np.cos(np.radians(center_lat))  # lon degree → m
    pixel_height_m = res_y * 110540                                     # lat degree → m
    print("--- Pixel Size Analysis ---")
    print(f"CRS: {crs.to_epsg()} (geographic — converting degrees to metres)")
    print(f"Resolution: {res_x:.6f}° × {res_y:.6f}°")
    print(f"Approx. metres: {pixel_width_m:.0f} m wide × {pixel_height_m:.0f} m tall")


## 5 — Load Elevation as Array

In [ ]:
# Two utility functions used throughout the pipeline:
# load_raster: reads a GeoTIFF band into a float32 numpy array, replacing nodata with NaN.
# resample_to_template: reprojects any raster to match the elevation grid exactly
#   (same CRS, extent, pixel size) so all layers align pixel-for-pixel.

def load_raster(path):
    with rasterio.open(path) as src:
        data = src.read(1).astype(np.float32)   # read band 1 as float32
        data[data == src.nodata] = np.nan        # replace nodata sentinel with NaN
        meta      = src.meta.copy()
        transform = src.transform   # affine transform: maps pixel indices to coordinates
        crs       = src.crs
    return data, meta, transform, crs


def resample_to_template(src_path, template_meta, resampling=Resampling.bilinear):
    """
    Reproject and resample any raster to match the elevation template grid.
    template_meta contains the target CRS, transform, width, and height.
    resampling=bilinear: weighted average of 4 nearest source pixels (smooth, good for continuous data).
    resampling=nearest:  picks the closest source pixel value (preserves categories, e.g. landuse).
    """
    with rasterio.open(src_path) as src:
        # Allocate an empty output array with the target grid dimensions
        dst = np.empty(
            (1, template_meta["height"], template_meta["width"]), dtype=np.float32
        )
        reproject(
            source=rasterio.band(src, 1),
            destination=dst[0],
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=template_meta["transform"],   # target pixel grid
            dst_crs=template_meta["crs"],               # target coordinate system
            resampling=resampling,
            dst_nodata=np.nan
        )
    return dst[0].astype(np.float32)



def resample_array(arr, source_meta, target_res, categorical=False):
    """
    Resample a 2D float32 array to target_res metres in-place using rasterio.
    source_meta must contain 'crs', 'transform', 'height', 'width'.
    categorical=True uses nearest-neighbour (preserves integer class codes).
    Returns (resampled_array, new_meta).
    """
    bounds = rasterio.transform.array_bounds(
        source_meta["height"], source_meta["width"], source_meta["transform"]
    )
    new_transform, new_w, new_h = calculate_default_transform(
        source_meta["crs"], source_meta["crs"],
        source_meta["width"], source_meta["height"],
        *bounds, resolution=target_res
    )
    dst = np.full((new_h, new_w), np.nan, dtype=np.float32)
    reproject(
        source=arr,
        destination=dst,
        src_transform=source_meta["transform"],
        src_crs=source_meta["crs"],
        dst_transform=new_transform,
        dst_crs=source_meta["crs"],
        resampling=Resampling.nearest if categorical else Resampling.bilinear,
        src_nodata=np.nan,
        dst_nodata=np.nan,
    )
    new_meta = source_meta.copy()
    new_meta.update({"width": new_w, "height": new_h, "transform": new_transform})
    return dst, new_meta

elev_data, elev_meta, elev_transform, elev_crs = load_raster(elev_masked)

# study_mask: True = outside study area / ocean — used to mask all layers consistently
study_mask = np.isnan(elev_data)

print(f"Elevation array: {elev_data.shape}  |  CRS: {elev_crs}")
print(f"Value range: {np.nanmin(elev_data):.0f} – {np.nanmax(elev_data):.0f} m")
print(f"study_mask: {study_mask.sum():,} NoData pixels ({100*study_mask.mean():.1f}% of grid)")

## Terrain Derivatives & Focal Kernel
Compute **slope**, **TRI** and **roughness** (3 × 3 focal SD) from the 100 m elevation layer.

In [ ]:
# Compute four terrain derivatives from the 100 m elevation raster.
#
# Slope   : gradient magnitude (degrees) — steeper terrain is less accessible.
# TPI     : Topographic Position Index — elevation relative to neighbourhood mean.
#           Positive = ridges, negative = valleys. Not used in the final model
#           (negative values incompatible with log1p; correlated with elevation).
# TRI     : Terrain Ruggedness Index — mean absolute difference from neighbours.
# Roughness: focal standard deviation of elevation (3×3 window).
#           TRI and Roughness both capture terrain heterogeneity but are computed
#           differently. Roughness is more stable in low-relief areas.

def compute_slope(elev, transform):
    """Slope in degrees from elevation array."""
    # np.gradient returns the finite-difference gradient along each axis
    # divided by the pixel spacing (transform[4]=y-spacing, transform[0]=x-spacing)
    dy, dx = np.gradient(elev, np.abs(transform[4]), np.abs(transform[0]))
    slope  = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))
    slope[np.isnan(elev)] = np.nan   # propagate NoData mask
    return slope.astype(np.float32)


def compute_tri(elev, size=3):
    """Terrain Ruggedness Index: mean absolute difference from neighbours."""
    half = size // 2
    # pad mirrors edge values to avoid border artefacts in the sliding window
    pad  = np.pad(np.nan_to_num(elev), half, mode="edge")
    tri  = np.zeros_like(elev)
    for di in range(-half, half + 1):
        for dj in range(-half, half + 1):
            if di == 0 and dj == 0:
                continue   # skip the centre pixel (comparing to itself = 0)
            # Shift the padded array to get the neighbour at offset (di, dj)
            shifted = pad[half + di: half + di + elev.shape[0],
                          half + dj: half + dj + elev.shape[1]]
            tri += np.abs(np.nan_to_num(elev) - shifted)
    tri = tri / (size * size - 1)   # average over all neighbours (8 for 3×3 window)
    tri[np.isnan(elev)] = np.nan
    return tri.astype(np.float32)


def im_kernel(elev, mw=3, stat="sd"):
    """Focal moving-window statistic (mean / sd / var) over VALID neighbours only.
    mw: window size in pixels (must be odd). mw=3 -> 300 m neighbourhood at 100 m.
    stat='sd': standard deviation of elevation within the window = Roughness.

    VALID NEIGHBOURS ONLY. The window sums are divided by the number of valid
    (non-NaN) pixels in the window, not by mw*mw. The earlier version filled NaN
    with 0 m before filtering, so every pixel on the border and the coast averaged
    real elevations with fictitious 0 m neighbours: 89,716 such pixels, a third of
    them above the interior 99.9th percentile, and the global maximum -- which the
    synthetic experiment's min-max scaling is anchored on -- came from them.
    Computed in float64: E[x^2] - E[x]^2 at elevations of ~3000 m loses the
    variance of flat terrain to float32 rounding.
    """
    assert mw >= 1 and mw % 2 == 1, "mw must be a positive odd integer"
    valid = np.isfinite(elev)
    x  = np.where(valid, elev, 0.0).astype(np.float64)
    n  = uniform_filter(valid.astype(np.float64), size=mw, mode="constant", cval=0.0)
    s1 = uniform_filter(x, size=mw, mode="constant", cval=0.0)
    with np.errstate(invalid="ignore", divide="ignore"):
        mean = s1 / n
        if stat == "mean":
            result = mean
        elif stat in ("sd", "var"):
            s2  = uniform_filter(x * x, size=mw, mode="constant", cval=0.0)
            var = np.maximum(s2 / n - mean ** 2, 0.0)   # clamp to 0 to avoid sqrt(-eps)
            result = np.sqrt(var) if stat == "sd" else var
        else:
            raise ValueError(f"Unsupported stat: {stat}")
    result = np.where(valid, result, np.nan)
    return result.astype(np.float32)


# Cache all four derivatives together — if any is missing, recompute all
_caches = {
    "slope_data":     f"{DATA_DIR}/layers/slope_{TARGET_RES}m.npy",
    "tri_data":       f"{DATA_DIR}/layers/tri_{TARGET_RES}m.npy",
    # new name: the old roughness_{res}m.npy was computed with 0 m outside Germany
    "roughness_data": f"{DATA_DIR}/layers/roughness_nanaware_{TARGET_RES}m.npy",
}

if all(os.path.exists(p) for p in _caches.values()):
    slope_data     = np.load(_caches["slope_data"])
    tri_data       = np.load(_caches["tri_data"])
    roughness_data = np.load(_caches["roughness_data"])
    print("Terrain derivatives: loaded from cache.")
else:
    slope_data     = compute_slope(elev_data, elev_transform)
    tri_data       = compute_tri(elev_data)
    roughness_data = im_kernel(elev_data, mw=3, stat="sd")   # 3x3 focal standard deviation
    for key, path in _caches.items():
        np.save(path, locals()[key])
    print("Terrain derivatives: computed and cached.")

for name, arr in [("Slope", slope_data), ("TRI", tri_data), ("Roughness", roughness_data)]:
    print(f"  {name:<12} min={np.nanmin(arr):8.2f}  max={np.nanmax(arr):8.2f}  "
          f"NaN={np.isnan(arr).sum()}")

## 6 — Environmental Variable Acquisition
Download and prepare all environmental covariates aligned to the 100 m UTM template.

| Layer | Source | Native resolution | Method |
|---|---|---|---|
| Winter temperature (BIO11 — coldest quarter) | CHELSA v2.1, 1981–2010 | 30″ (~1 km) | vsicurl window, bilinear; stored in 0.1 K |
| Summer precipitation (BIO18 — warmest quarter) | CHELSA v2.1, 1981–2010 | 30″ (~1 km) | vsicurl window, bilinear; stored in 0.1 mm |
| NDVI | pre-processed composite `ndvi_germany_2020_30m.tif` (see 6h) | 30 m | bilinear |
| Population | `population_germany_2020.tif` | raster | bilinear |
| Road density | OSM (Geofabrik extracts, June 2026) | vector | rasterize + 5×5 focal mean |
| Railway density | OSM (active lines only) | vector | rasterize + 5×5 focal mean |
| Distance to water | OSM water bodies + rivers/streams/canals | vector | rasterize + Euclidean distance |
| Habitat score (Landuse) | ESA WorldCover 2021 | 10 m | majority (mode), class → score 0–5 |
| PA distance / PA density | WDPA/WDOECM (June 2026) | vector | rasterize + distance / 50×50 focal mean |


In [ ]:
# ============================================================
# STEP 6 — OSM Data Paths & Shared Loader
# ============================================================
# OpenStreetMap (OSM) data was downloaded as per-Bundesland/Bezirk Geofabrik
# extracts and merged here to cover all of Germany.

# BASE_PATH: root folder containing all OSM subdirectories
BASE_PATH = os.path.join(PROJECT_ROOT, "data.nosync/OSM")

# One entry per downloaded Geofabrik regional extract.
# load_osm reads the same shapefile from every directory and concatenates the results.
OSM_DIRS = [
    "arnsberg-regbez-260616-free",
    "berlin-260615-free",
    "brandenburg-260615-free",
    "bremen-260615-free",
    "detmold-regbez-260616-free",
    "duesseldorf-regbez-260616-free",
    "freiburg-regbez-260615-free",
    "hamburg-260615-free",
    "hessen-260615-free",
    "karlsruhe-regbez-260615-free",
    "koeln-regbez-260617-free",
    "mecklenburg-vorpommern-260615-free",
    "mittelfranken-260615-free",
    "muenster-regbez-260616-free",
    "niederbayern-260615-free",
    "niedersachsen-260615-free",
    "oberbayern-260615-free",
    "oberfranken-260622-free",
    "oberpfalz-260615-free",
    "rheinland-pfalz-260616-free",
    "saarland-260617-free",
    "sachsen-260616-free",
    "sachsen-anhalt-260616-free",
    "schleswig-holstein-260616-free",
    "schwaben-260615-free",
    "stuttgart-regbez-260622-free",
    "thueringen-260616-free",
    "tuebingen-regbez-260615-free",
    "unterfranken-260615-free",
]

def load_osm(base_path, dirs, filename, filter_col=None, keep_vals=None):
    """
    Read and merge one OSM shapefile from each directory in `dirs`.

    base_path   : root data folder
    dirs        : list of subdirectory names (one per OSM extract)
    filename    : shapefile name to look for inside each subdirectory
    filter_col  : optional column name to filter rows (e.g. 'fclass')
    keep_vals   : list of values to keep in filter_col (e.g. ['river', 'stream'])

    Returns a merged GeoDataFrame, or None if no files were found.
    """
    gdfs = []   # list to collect GeoDataFrames from each region
    for d in dirs:
        p = os.path.join(base_path, d, filename)   # full path to this region's shapefile
        if os.path.exists(p):
            gdfs.append(gpd.read_file(p))   # read if file exists
        else:
            print(f"  Missing: {os.path.basename(d)}/{filename}")
    if not gdfs:
        return None   # no data found in any region

    # pd.concat stacks all regional DataFrames vertically into one table
    # ignore_index=True resets the row index so there are no duplicate indices
    merged = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)

    # Optional row filter: keep only rows where filter_col is in keep_vals
    if filter_col and keep_vals:
        merged = merged[merged[filter_col].isin(keep_vals)]
    return merged


# 6a - Winter Temperature — Coldest Quarter (CHELSA BIO11)

In [ ]:
# ============================================================
# STEP 6a — Coldest-Quarter Temperature (CHELSA BIO11)
# ============================================================
# CHELSA provides climate rasters at 1 km resolution globally.
# BIO11 = mean temperature of the coldest quarter.
# STORAGE UNIT: 0.1 K (CHELSA V2.1, scale 0.1, offset -273.15), i.e.
# degC = x * 0.1 - 273.15. The GeoTIFF window written below carries no
# scale/offset metadata, so rasterio reads the raw integers; the pipeline keeps
# them and converts only for figures.
# We read only the bounding box window of our study area using
# rasterio's vsicurl streaming — no full 5 GB download needed.

def download_chelsa_bio11(gdf, path=os.path.join(DATA_DIR, "map_data")):
    from rasterio.windows import from_bounds as window_from_bounds
    out_path = os.path.join(path, "chelsa_bio11_germany.tif")
    if os.path.exists(out_path):   # skip if already downloaded
        print(f"CHELSA BIO11 already present: {out_path}")
        return out_path

    # /vsicurl/ prefix allows rasterio to stream from a remote URL without
    # downloading the whole file — it reads only the requested window
    url = ("/vsicurl/https://os.zhdk.cloud.switch.ch/chelsav2/GLOBAL/"
           "climatologies/1981-2010/bio/CHELSA_bio11_1981-2010_V.2.1.tif")

    # Get the bounding box of the study area in WGS84 (the CHELSA raster's CRS)
    gdf_wgs84 = gdf.to_crs("EPSG:4326")
    minx, miny, maxx, maxy = gdf_wgs84.total_bounds
    print(f"Reading CHELSA BIO11 window ({minx:.2f},{miny:.2f} – {maxx:.2f},{maxy:.2f})...")
    with rasterio.open(url) as src:
        # from_bounds converts geographic coordinates to a rasterio Window object
        # (row/column indices inside the raster) — only that region is downloaded
        win  = window_from_bounds(minx, miny, maxx, maxy, src.transform)
        data = src.read(1, window=win).astype(np.float32)   # read band 1 within window
        tr   = src.window_transform(win)   # get the transform for just this window
        profile = src.profile.copy()
        profile.update({"height": data.shape[0], "width": data.shape[1],
                        "transform": tr, "driver": "GTiff",
                        "dtype": "float32", "compress": "lzw", "count": 1})
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(data, 1)   # write data as band 1
    print(f"Saved CHELSA BIO11 → {out_path}  ({data.shape[0]}×{data.shape[1]} px)")
    return out_path

temp_path = download_chelsa_bio11(regions)

# CACHE NAME CARRIES THE VARIABLE. It used to be "temp_{res}m.npy", a name the
# pipeline had already used for CHELSA BIO1 (ANNUAL mean temperature) before the
# switch to BIO11. The existence check below then found that old BIO1 file and
# loaded it silently -- the BIO11 GeoTIFF was downloaded but never used, and
# every "WinterTemp" value was in fact the annual mean. A variable-specific name
# makes that collision impossible; the old temp_{res}m.npy is simply ignored.
_temp_cache = f"{DATA_DIR}/layers/chelsa_bio11_{TARGET_RES}m.npy"
if os.path.exists(_temp_cache):
    temp_data = np.load(_temp_cache)
    print(f"Temperature: loaded from cache ({_temp_cache})")
else:
    temp_data = resample_to_template(temp_path, elev_meta)
    temp_data[study_mask] = np.nan
    np.save(_temp_cache, temp_data)
    print(f"Temperature: resampled to {TARGET_RES} m UTM and cached -> {_temp_cache}")

# PLAUSIBILITY GUARD against exactly that failure. The coldest-quarter mean in
# Germany sits around 1 degC (measured: median 0.94, p1 -2.65, p99 3.08); the
# annual mean sits around 9 degC. A median above 5 degC means the wrong variable.
_t_med = float(np.nanmedian(temp_data)) * 0.1 - 273.15
assert _t_med < 5.0, (
    f"WinterTemp median is {_t_med:.2f} degC -- that is an ANNUAL mean, not the "
    f"coldest quarter. Delete {_temp_cache} and re-run this cell.")
print(f"WinterTemp (BIO11) plausible: median {_t_med:.2f} degC")

print("\nLocal data base path:", BASE_PATH)
print("OSM directories:", OSM_DIRS)


## 6b — Population Density
*(Layer excluded from the final set at variable selection — kept here for completeness)*

Loads `population_germany_2020.tif` and resamples it bilinearly to the 100 m template. Pixels inside Germany without a population value are set to 0 (uninhabited); pixels outside stay NaN.


In [ ]:
# ============================================================
# STEP 6b — Population Density
# ============================================================
# Human population density is an important predictor for wild boar:
# high human presence tends to reduce habitat suitability (hunting
# pressure, habitat fragmentation) but urban edges can also attract boar.
# We try WorldPop (constrained 2020) first, then GHS-POP as fallback.

def load_population(base_path, elev_meta, study_mask):
    from rasterio.merge import merge   # import here to avoid circular issues at top level

    H, W  = elev_meta["height"], elev_meta["width"]   # target raster dimensions

    pop_path = os.path.join(PROJECT_ROOT, "data.nosync/population_germany_2020.tif")
    if os.path.exists(pop_path):
        print(f"Loading population: {pop_path}")
        arr = resample_to_template(pop_path, elev_meta, resampling=Resampling.bilinear)
        arr[arr < 0] = 0
        arr[study_mask] = np.nan
        return arr

    print("  [placeholder] No population file found — returning NaN array")
    return np.full((H, W), np.nan, dtype=np.float32)

_pop_cache = f"{DATA_DIR}/layers/pop_{TARGET_RES}m.npy"
if os.path.exists(_pop_cache):
    pop_data = np.load(_pop_cache)
    print(f"Population: loaded from cache ({_pop_cache})")
else:
    pop_data = load_population(BASE_PATH, elev_meta, study_mask)
    # NaN pixels inside Germany where no population is recorded are real zeros (uninhabited),
    # not missing data -- replace with 0. Pixels outside Germany stay NaN.
    pop_data = np.where(study_mask, np.nan, np.nan_to_num(pop_data, nan=0.0))
    np.save(_pop_cache, pop_data)
    print(f"Population: resampled and cached -> {_pop_cache}")


total = np.nansum(pop_data)   # total population across all pixels
print(f"Population: min={np.nanmin(pop_data):.1f}  max={np.nanmax(pop_data):.0f}  "
      f"total={total/1e6:.1f} M persons")
print(f"Reference (Germany, ~2020): ~83.2 M  →  diff: {total/1e6 - 83.2:+.1f} M")


## 6c — Road Density (OSM)
Rasterize all OSM roads from the Geofabrik extracts listed in step 6 (every German state or Regierungsbezirk) as presence/absence, then apply a **5×5 focal mean** (= 500 m window) — equivalent to R `focal(w=5, fun="mean")`. *Limitation:* the extracts stop at the border, so within 2 px of it the density is slightly underestimated (outside counts as road-free).

In [ ]:
from pathlib import Path
# Rasterize linear OSM features (roads, railways) and compute focal density.
# A 5×5 uniform filter (window=5) counts how many pixels in the 500×500 m
# neighbourhood are covered by a road/railway — a simple proxy for network density.
# Results are cached as .npy files to avoid recomputing on every notebook run.

def compute_linear_density(base_path, dirs, elev_meta, study_mask,
                            shp_name, label="features", window=5,
                            exclude_fclass=None):
    """Rasterize linear OSM features region-by-region to avoid loading all geometries at once."""
    H, W      = elev_meta["height"], elev_meta["width"]
    transform = elev_meta["transform"]

    # Accumulate a single binary raster across all regions.
    # Processing one directory at a time keeps peak RAM to one region's geometries.
    raster        = np.zeros((H, W), dtype=np.float32)
    total_features = 0
    found_any      = False

    for d in dirs:
        p = os.path.join(base_path, d, shp_name)
        if not os.path.exists(p):
            print(f"  Missing: {os.path.basename(d)}/{shp_name}")
            continue
        found_any = True

        region = gpd.read_file(p)

        if exclude_fclass is not None and "fclass" in region.columns:
            region = region[~region["fclass"].isin(exclude_fclass)]

        region = region.to_crs(elev_meta["crs"])

        shapes = [(geom, 1) for geom in region.geometry
                  if geom is not None and not geom.is_empty]

        if shapes:
            tile = rio_rasterize(shapes, out_shape=(H, W), transform=transform,
                                 fill=0, dtype=np.float32, all_touched=True)
            np.maximum(raster, tile, out=raster)   # OR: pixel = 1 if any region covers it
            total_features += len(shapes)

        del region, shapes   # free memory before loading next region

    if not found_any:
        # FAIL, do not return a placeholder. An all-NaN layer used to be returned
        # and cached here, and the stack, the collinearity analysis and the
        # checkpoint would then have carried an empty covariate without any error.
        raise FileNotFoundError(
            f"{shp_name} not found in any of the {len(dirs)} OSM directories under "
            f"{base_path} -- download the Geofabrik extracts listed in OSM_DIRS first.")

    print(f"  Active {label}: {total_features:,}")

    # uniform_filter computes the local mean within a (window × window) neighbourhood.
    # For a binary raster this equals the fraction of pixels covered by the feature —
    # i.e. a simple linear density estimate within a 500 m window.
    #
    # KNOWN LIMITATION (kept deliberately): the OSM extracts stop at the German
    # border, so pixels outside count as road-free in this window mean. Within
    # 2 px of the border the density is therefore underestimated (measured: 0.434
    # in the outermost pixel row vs 0.487 inside; 151,000 px = 0.42% of the land).
    # A mean over inside pixels only would need the binary road raster, which is
    # not cached, and the June-2026 extracts are no longer available. In the
    # synthetic experiment no sampled point lies within 16 px of the border (patch
    # buffer), so the affected strip appears only at the edge of some CNN patches.
    density = uniform_filter(raster, size=window)
    density[study_mask] = np.nan   # mask pixels outside the study area
    return density

# Cache check: load pre-computed road density if it exists,
# otherwise run the rasterization and save the result.
_cache = str(Path.home() / "wolf_sdm_stack" / f"road_{TARGET_RES}m.npy")
if os.path.exists(_cache):
    road_data = np.load(_cache)
    print(f"Road density: loaded from cache ({_cache})")
else:
    road_data = compute_linear_density(
        BASE_PATH, OSM_DIRS, elev_meta, study_mask,
        shp_name="gis_osm_roads_free_1.shp", label="roads"
    )
    np.save(_cache, road_data)
    print(f"Road density: computed and cached → {_cache}")
print(f"Road density: max={np.nanmax(road_data):.3f}  mean={np.nanmean(road_data):.4f}")


## 6d — Railway Density (OSM)
Active railways only (excludes `abandoned`, `disused`, `construction`).
Same rasterize + 5×5 focal mean as roads.

In [ ]:
# Same approach as road density, but for railway lines.
# Abandoned and disused lines are excluded — they no longer act as barriers.

_cache = f"{DATA_DIR}/layers/railway_{TARGET_RES}m.npy"
if os.path.exists(_cache):
    railway_data = np.load(_cache)
    print(f"Railway density: loaded from cache ({_cache})")
else:
    railway_data = compute_linear_density(
        BASE_PATH, OSM_DIRS, elev_meta, study_mask,
        shp_name="gis_osm_railways_free_1.shp", label="railways",
        exclude_fclass=["abandoned", "disused", "construction"]
    )
    np.save(_cache, railway_data)
    print(f"Railway density: computed and cached → {_cache}")
print(f"Railway density: max={np.nanmax(railway_data):.3f}  mean={np.nanmean(railway_data):.4f}")


## 6e — Distance to Water (OSM)
Rasterize water bodies and major waterways (river/stream/canal),
then compute the **Euclidean distance transform** — equivalent to R `distance()`.

In [ ]:
# Compute Euclidean distance to nearest water body or waterway (rivers, streams, canals).
# Uses scipy distance_transform_edt: first rasterizes all water features to a binary mask,
# then computes the per-pixel distance to the nearest water pixel.
# Wild boar use water for drinking and wallowing, so proximity to water is
# expected to be a positive habitat predictor.

def compute_water_distance(base_path, dirs, elev_meta, study_mask):
    H, W      = elev_meta["height"], elev_meta["width"]
    transform = elev_meta["transform"]
    pixel_m   = abs(transform.a)

    # Accumulate binary water mask one region at a time to limit peak RAM usage.
    water_mask_r = np.zeros((H, W), dtype=np.uint8)
    total_features = 0

    water_classes = ["river", "stream", "canal"]

    for d in dirs:
        region_shapes = []

        # Polygonal water bodies (lakes, reservoirs, ponds)
        p_water = os.path.join(base_path, d, "gis_osm_water_a_free_1.shp")
        if os.path.exists(p_water):
            gdf = gpd.read_file(p_water).to_crs(elev_meta["crs"])
            region_shapes += [(g, 1) for g in gdf.geometry if g is not None and not g.is_empty]
            del gdf

        # Linear waterways — rivers, streams, canals only
        p_ways = os.path.join(base_path, d, "gis_osm_waterways_free_1.shp")
        if os.path.exists(p_ways):
            gdf = gpd.read_file(p_ways)
            if "fclass" in gdf.columns:
                gdf = gdf[gdf["fclass"].isin(water_classes)]
            gdf = gdf.to_crs(elev_meta["crs"])
            region_shapes += [(g, 1) for g in gdf.geometry if g is not None and not g.is_empty]
            del gdf

        if region_shapes:
            tile = rio_rasterize(region_shapes, out_shape=(H, W), transform=transform,
                                 fill=0, dtype=np.uint8, all_touched=True)
            np.maximum(water_mask_r, tile, out=water_mask_r)
            total_features += len(region_shapes)

        del region_shapes

    if total_features == 0:
        print("  [placeholder] Water shapefiles not found")
        return np.full((H, W), np.nan, dtype=np.float32)

    print(f"  Water features (bodies + waterways): {total_features:,}")

    # distance_transform_edt: for every pixel with value 0 (non-water),
    # returns the Euclidean distance in pixels to the nearest pixel with value 1 (water).
    dist_px = distance_transform_edt(1 - water_mask_r)
    dist_m  = (dist_px * pixel_m).astype(np.float32)
    dist_m[study_mask] = np.nan
    return dist_m

_cache = f"{DATA_DIR}/layers/water_{TARGET_RES}m.npy"
if os.path.exists(_cache):
    water_data = np.load(_cache)
    print(f"Water distance: loaded from cache ({_cache})")
else:
    water_data = compute_water_distance(BASE_PATH, OSM_DIRS, elev_meta, study_mask)
    np.save(_cache, water_data)
    print(f"Water distance: computed and cached → {_cache}")
print(f"Water distance: max={np.nanmax(water_data)/1000:.1f} km  "
      f"mean={np.nanmean(water_data)/1000:.1f} km")


## 6f — Habitat Score from ESA WorldCover (layer name `Landuse`)
ESA WorldCover 2021 (10 m) is reprojected to the 100 m grid by majority (mode) — each 100 m cell gets the class that covers most of it — and each class is mapped to an a-priori *Sus scrofa* habitat score (0–5). The scores are a hypothesis, not a derived result — see the citation notes in the code.

| ESA WorldCover class | Score | Rationale |
|---|---|---|
| 10 Tree cover | 5 | Primary shelter and mast |
| 20 Shrubland, 40 Cropland | 4 | Cover and foraging; crop raiding |
| 30 Grassland, 90 Herbaceous wetland, 95 Mangroves | 3 | Foraging / wallowing, limited cover |
| — | 2 | (no class assigned) |
| 50 Built-up, 60 Bare / sparse vegetation, 100 Moss and lichen | 1 | Avoided but crossed |
| 70 Snow and ice, 80 Permanent water bodies | 0 | Not occupiable |

**Not a model predictor.** The synthetic experiment uses only score 5 (tree cover), as the binary forest field W of the configurational data-generating process; the layer is excluded from Pearson/VIF and analysed with Kruskal–Wallis instead.


In [ ]:
# ── ESA WorldCover 2021 — load, reproject, mask, NaN check ─────────────────
# The Germany export comes as two 10 m GeoTIFF tiles, already in UTM 32N and
# stacked on one 10 m grid. Steps: join the tiles in a VRT → majority-resample
# to the elevation template (UTM 32N, 100 m) → apply study_mask → coverage check.

import glob as _glob
from xml.sax.saxutils import escape as _xml_escape

WC_CLASS = {
    10: "Tree cover",
    20: "Shrubland",
    30: "Grassland",
    40: "Cropland",
    50: "Built-up",
    60: "Bare / sparse vegetation",
    70: "Snow and ice",
    80: "Permanent water bodies",
    90: "Herbaceous wetland",
    95: "Mangroves",
   100: "Moss and lichen",
}
WC_NODATA = 0   # ESA WorldCover uses 0 as the nodata sentinel

# "mode" in the name: the old worldcover_raw_{res}m.npy was nearest-neighbour
WC_CACHE = f"{DATA_DIR}/layers/worldcover_mode_{TARGET_RES}m.npy"

if os.path.exists(WC_CACHE):
    worldcover_data = np.load(WC_CACHE)
    print(f"WorldCover: loaded from cache ({WC_CACHE})")
else:
    WC_DATA_DIR = os.path.join(PROJECT_ROOT, "data.nosync")
    tile_paths = sorted(_glob.glob(os.path.join(WC_DATA_DIR, "worldcover_germany_2021_10m-*.tif")))
    if not tile_paths:
        raise FileNotFoundError(f"No WorldCover tiles found in {WC_DATA_DIR}")
    print(f"Found {len(tile_paths)} WorldCover tiles -- joining them in a VRT...")

    # JOIN THE TILES IN A VRT, NOT IN MEMORY. The tiles hold 5.5 billion 10 m
    # pixels; merging them into one array and copying it into a MemoryFile
    # needed ~11 GB of RAM. A VRT is a few lines of XML that GDAL reads tile by
    # tile. It also makes the mode exact at the tile seam, which does not fall on
    # a 100 m row of the template: resampling the tiles one at a time would take
    # that row's majority from only part of its pixels.
    # NODATA = 0 IS DECLARED. The tiles carry no nodata value, and WorldCover's
    # fill value 0 (outside the export area) is not a class. Under nearest
    # neighbour that did not matter (0 became NaN afterwards); under MODE an
    # undeclared 0 would be counted as a class and could win the majority in
    # cells along the border.
    _tiles = []
    for _p in tile_paths:
        with rasterio.open(_p) as _s:
            assert _s.crs == elev_meta["crs"] and abs(_s.transform.a - 10) < 1e-9 \
                and abs(_s.transform.e + 10) < 1e-9, f"unexpected grid in {_p}"
            _tiles.append((_p, _s.transform.c, _s.transform.f, _s.width, _s.height, _s.crs.to_wkt()))
    _x0 = min(t[1] for t in _tiles); _y0 = max(t[2] for t in _tiles)
    _W  = int(round((max(t[1] + 10 * t[3] for t in _tiles) - _x0) / 10))
    _H  = int(round((_y0 - min(t[2] - 10 * t[4] for t in _tiles)) / 10))
    _sources = "".join(
        f'<SimpleSource><SourceFilename relativeToVRT="0">{_xml_escape(p)}</SourceFilename>'
        f'<SourceBand>1</SourceBand>'
        f'<SrcRect xOff="0" yOff="0" xSize="{w}" ySize="{h}"/>'
        f'<DstRect xOff="{int(round((x - _x0) / 10))}" yOff="{int(round((_y0 - y) / 10))}" '
        f'xSize="{w}" ySize="{h}"/></SimpleSource>'
        for p, x, y, w, h, _ in _tiles)
    _vrt = (f'<VRTDataset rasterXSize="{_W}" rasterYSize="{_H}">'
            f'<SRS>{_xml_escape(_tiles[0][5])}</SRS>'
            f'<GeoTransform>{_x0}, 10, 0, {_y0}, 0, -10</GeoTransform>'
            f'<VRTRasterBand dataType="Byte" band="1"><NoDataValue>{WC_NODATA}</NoDataValue>'
            f'{_sources}</VRTRasterBand></VRTDataset>')
    print(f"VRT: {_W} x {_H} px at 10 m")

    # Reproject + resample to the elevation template grid (UTM 32N, 100 m).
    # MAJORITY (mode), not nearest neighbour. Nearest neighbour takes the
    # class of ONE 10 m pixel out of the ~100 in each 100 m cell, so a cell
    # that is 30% forest becomes "forest" with probability ~0.3 -- speckle
    # that shows up as spurious forest edges, i.e. exactly in the edge
    # density the configurational arm is built on. Mode returns the class
    # covering most of the cell and keeps integer class codes.
    worldcover_data = resample_to_template(_vrt, elev_meta, resampling=Resampling.mode)

    # Replace nodata sentinel with NaN before masking
    worldcover_data[worldcover_data == WC_NODATA] = np.nan
    # Apply study-area mask (pixels outside Germany → NaN)
    worldcover_data[study_mask] = np.nan

    np.save(WC_CACHE, worldcover_data)
    print(f"WorldCover: reprojected, masked, cached → {WC_CACHE}")

# ── Coverage / NaN check ────────────────────────────────────────────────────
# study_mask is True outside the study area → invert it to get study-area pixels.
# Any NaN *inside* the study area is a genuine WorldCover data gap.

inside_mask   = ~study_mask                              # True = inside Germany
study_px      = int(inside_mask.sum())                   # total study-area pixels
gap_px        = int(np.isnan(worldcover_data[inside_mask]).sum())  # NaN inside Germany
covered_px    = study_px - gap_px                        # pixels with actual data
outside_px    = int(study_mask.sum())                    # pixels outside Germany
total_px      = worldcover_data.size

print(f"Total raster pixels      : {total_px:>12,}")
print(f"Outside study area (NaN) : {outside_px:>12,}  ({outside_px / total_px * 100:.1f}% of grid)")
print(f"Inside study area        : {study_px:>12,}  ({study_px / total_px * 100:.1f}% of grid)")
print()
print("Coverage within study area:")
print(f"  WorldCover data present : {covered_px:>10,}  ({covered_px / study_px * 100:.2f}%)")
print(f"  Gaps (NaN, no WC data)  : {gap_px:>10,}  ({gap_px / study_px * 100:.2f}%)")
print()
if gap_px / study_px < 0.01:
    print("Coverage assessment: GOOD — gaps < 1%, layer is suitable for analysis.")
elif gap_px / study_px < 0.05:
    print("Coverage assessment: ACCEPTABLE — gaps between 1–5%, consider gap-filling.")
else:
    print("Coverage assessment: POOR — gaps > 5%, investigate missing tiles.")
print()
print("WorldCover class distribution inside study area:")
for code, label in WC_CLASS.items():
    n = int((worldcover_data == code).sum())
    if n > 0:
        print(f"  {code:>3}  {label:<28} {n:>10,}  ({n / covered_px * 100:.1f}%)")


# ── Sus scrofa habitat score (0–5) ─────────────────────────────────────────
# WorldCover class codes are mapped to an a-priori 0–5 Sus scrofa habitat score.

WC_SCORE = {
    # ── Sus scrofa-specific habitat suitability score (0 = unusable, 5 = prime) ──
    # !! PROVISIONAL — each score level needs a citable source in the methods !!
    # This is an a priori suitability hypothesis, NOT a derived result.
    # It is NOT a model predictor: the synthetic experiment uses only score 5
    # (tree cover), as the binary forest field W. It is excluded from Pearson/VIF
    # and from every dissimilarity computation.
    #
    # Score 5 — Prime habitat: forest cover alone (mast, shelter, acorns/beech mast)
    # CITATION NEEDED: e.g. Schley & Roper (2003) Mammal Rev on Sus scrofa food habits
    10: 5,   # Tree cover       — primary shelter; acorn/beech mast = key food source
    #
    # Score 4 — High-value secondary habitat: cropland (crop raiding) + scrub (cover/foraging)
    # CITATION NEEDED: e.g. Geisser & Reyer (2004) J Appl Ecol on crop damage;
    #                  Massei & Genov (2004) on habitat breadth
    40: 4,   # Cropland         — maize/cereal foraging; second only to forest in value
    20: 4,   # Shrubland        — cover + foraging, often transitional with forest
    #
    # Score 3 — Moderate: wetland (wallowing/riparian edge) and open grassland (forage, exposed)
    # CITATION NEEDED: e.g. Nores et al. (2008) Galemys on wetland use;
    #                  Truvé & Lemel (2003) on grassland vs. forest preference
    90: 3,   # Herbaceous wetland — wallowing sites, riparian edge; limited cover
    95: 3,   # Mangroves         — dense natural vegetation (absent from study area)
    30: 3,   # Grassland         — foraging but open/exposed; less preferred than forest-edge
    #
    # Score 1 — Unsuitable but traversed: built/bare environments are avoided but crossed
    # CITATION NEEDED: e.g. Truvé & Lemel (2003) Wildl Biol on urban avoidance
    50: 1,   # Built-up          — generally avoided; some tolerance in peri-urban fringe
    60: 1,   # Bare/sparse veg   — minimal resources
   100: 1,   # Moss and lichen   — minimal resources
    #
    # Score 0 — Impassable / occupancy impossible
    # CITATION NEEDED: confirm open-water vs. riparian-edge distinction
    80: 0,   # Permanent water   — open water bodies; boar drink at edge but do not occupy
    70: 0,   # Snow and ice      — impassable barrier
}

_cache = f"{DATA_DIR}/layers/landuse_score_mode_{TARGET_RES}m.npy"   # from the mode-resampled WorldCover

if os.path.exists(_cache):
    landuse_data = np.load(_cache)
    print(f"Landuse score: loaded from cache ({_cache})")
else:
    # Start with all NaN so gaps and out-of-mask pixels remain NaN automatically
    landuse_data = np.full_like(worldcover_data, np.nan)
    for code, score in WC_SCORE.items():
        landuse_data[worldcover_data == code] = score
    np.save(_cache, landuse_data)
    print(f"Landuse score: computed and cached → {_cache}")
    # ── Self-assert: must contain only 0-5, no raw ESA codes ─────────────────
    _lu_valid = landuse_data[~np.isnan(landuse_data)]
    assert set(_lu_valid.astype(int).tolist()) <= {0,1,2,3,4,5}, \
        f"ASSERT FAILED: unexpected values in landuse_score: {set(_lu_valid.astype(int).tolist()) - {0,1,2,3,4,5}}"
    assert not np.isin(landuse_data, [10,20,30,40,50,60,70,80,90,95,100]).any(), \
        "ASSERT FAILED: raw ESA codes still present in landuse_score layer!"
    print("ASSERT PASSED: landuse_score contains only values {0,1,2,3,4,5} — no raw codes survive.")

print(f"Landuse (Sus scrofa score): min={np.nanmin(landuse_data):.0f}  "
      f"max={np.nanmax(landuse_data):.0f}  "
      f"mean={np.nanmean(landuse_data):.2f}")
print()
print("Score distribution inside study area:")
for score in range(6):
    n = int((landuse_data == score).sum())
    if n > 0:
        print(f"  score {score}: {n:>10,}  ({n / covered_px * 100:.1f}%)")


## 6g — Protected Area Distance & Density (WDPA)
Rasterise **WDPA / WDOECM** (World Database on Protected Areas) polygons to derive two layers:

| Layer | Description |
|-------|-------------|
| `PA_Distance` | Euclidean distance (m) to the nearest protected area (0 inside a PA) |
| `PA_Density`  | Fraction of a 50 × 50 px (~5 km) window covered by PAs |

**Data source:** UNEP-WCMC / IUCN — [protectedplanet.net](https://www.protectedplanet.net/)
**Release:** WDPA_WDOECM June 2026, Germany (`WDPA_WDOECM_Jun2026_Public_DEU`) — three shapefile splits merged into `data.nosync/map_data/WDPA_Germany.gpkg`.

All IUCN categories are included (I–IV strictly protected; V–VI and unclassified allow partial use). To restrict to strict protection, filter inside `compute_pa_layers`.

In [ ]:
# ============================================================
# STEP 6g — Protected Area Distance & Density (WDPA)
# ============================================================
# Data: WDPA/WDOECM June 2026 release, Germany — downloaded from protectedplanet.net
# The download splits the shapefile across three subfolders (_0, _1, _2).
# This cell merges the polygon splits, saves a single GeoPackage to
# data.nosync/map_data/WDPA_Germany.gpkg, then computes:
#   PA_Distance : Euclidean distance (m) to the nearest protected area (0 inside)
#   PA_Density  : fraction of a 50×50 px (~5 km) window covered by PAs

WDPA_SHP_DIR = os.path.join(
    PROJECT_ROOT, "data.nosync", "WDPA_WDOECM_Jun2026_Public_DEU_shp")
WDPA_STEM = "WDPA_WDOECM_Jun2026_Public_DEU_shp"


def load_wdpa_splits(shp_dir, stem, n_splits=3):
    """Merge polygon shapefiles from all WDPA split folders into one GeoDataFrame."""
    gdfs = []
    for i in range(n_splits):
        folder = os.path.join(shp_dir, f"{stem}_{i}")
        path   = os.path.join(folder,  f"{stem}-polygons.shp")
        if os.path.exists(path):
            gdfs.append(gpd.read_file(path))
            print(f"  Split {i}: {len(gdfs[-1]):,} polygons")
        else:
            print(f"  Split {i}: not found ({path})")
    if not gdfs:
        raise FileNotFoundError(f"No polygon shapefiles found under {shp_dir}")
    merged = gpd.GeoDataFrame(
        pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs
    )
    merged = merged[merged.geometry.notna() & ~merged.geometry.is_empty]
    return merged


def compute_pa_layers(wdpa_gdf, elev_meta, study_mask, density_window=50):
    """
    Rasterise WDPA polygons and compute:
      PA_Distance : Euclidean distance (m) to the nearest protected-area boundary.
                    Pixels inside a PA get distance 0.
      PA_Density  : Fraction of a density_window x density_window pixel window
                    covered by any protected area (default: 50 px = 5 km at 100 m).
    To use only strictly protected areas (IUCN I-IV), filter wdpa_gdf before
    calling this function:
        strict = wdpa_gdf[wdpa_gdf['IUCN_CAT'].isin(['Ia','Ib','II','III','IV'])]
        pa_dist, pa_dens = compute_pa_layers(strict, elev_meta, study_mask)
    """
    H, W      = elev_meta["height"], elev_meta["width"]
    transform = elev_meta["transform"]
    pixel_m   = abs(transform.a)

    wdpa_utm = wdpa_gdf.to_crs(elev_meta["crs"])
    wdpa_utm = wdpa_utm[wdpa_utm.geometry.notna() & ~wdpa_utm.geometry.is_empty]
    print(f"  PA features with valid geometry: {len(wdpa_utm):,}")

    shapes = [(g, 1) for g in wdpa_utm.geometry]

    # Binary raster: 1 = inside any PA, 0 = outside
    pa_mask = rio_rasterize(
        shapes, out_shape=(H, W), transform=transform,
        fill=0, dtype=np.uint8, all_touched=True,
    )

    # Distance to nearest PA boundary (metres)
    dist_px   = distance_transform_edt(1 - pa_mask)
    pa_dist_m = (dist_px * pixel_m).astype(np.float32)
    pa_dist_m[study_mask] = np.nan

    # Focal PA density: local mean of binary mask over density_window pixels
    pa_density = uniform_filter(pa_mask.astype(np.float32), size=density_window)
    pa_density[study_mask] = np.nan

    pct_covered = float(pa_mask.sum()) / float((~study_mask).sum()) * 100
    print(f"  Land area covered by PAs  : {pct_covered:.1f}%")
    print(f"  PA_Distance: max={np.nanmax(pa_dist_m)/1000:.1f} km  "
          f"mean={np.nanmean(pa_dist_m)/1000:.1f} km")
    print(f"  PA_Density : max={np.nanmax(pa_density):.3f}  "
          f"mean={np.nanmean(pa_density):.3f}")
    return pa_dist_m, pa_density


# ── Cache paths ───────────────────────────────────────────────────────────────
_cache_dist    = f"{DATA_DIR}/layers/pa_distance_{TARGET_RES}m.npy"
_cache_density = f"{DATA_DIR}/layers/pa_density_{TARGET_RES}m.npy"
_wdpa_gpkg     = f"{DATA_DIR}/map_data/WDPA_Germany.gpkg"

if os.path.exists(_cache_dist) and os.path.exists(_cache_density):
    pa_distance_data = np.load(_cache_dist)
    pa_density_data  = np.load(_cache_density)
    print(f"PA distance : loaded from cache ({_cache_dist})")
    print(f"PA density  : loaded from cache ({_cache_density})")
else:
    # Load from the downloaded WDPA Germany shapefiles (or cached GeoPackage)
    if os.path.exists(_wdpa_gpkg):
        print(f"Loading WDPA from merged cache: {_wdpa_gpkg}")
        _wdpa = gpd.read_file(_wdpa_gpkg)
    else:
        print("Merging WDPA Germany polygon splits...")
        _wdpa = load_wdpa_splits(WDPA_SHP_DIR, WDPA_STEM)
        _wdpa.to_file(_wdpa_gpkg, driver="GPKG")
        print(f"Merged {len(_wdpa):,} polygons saved -> {_wdpa_gpkg}")

    if "IUCN_CAT" in _wdpa.columns:
        strict_cats = {"Ia", "Ib", "II", "III", "IV"}
        n_strict  = int(_wdpa["IUCN_CAT"].isin(strict_cats).sum())
        n_partial = len(_wdpa) - n_strict
        print(f"  Strictly protected (IUCN I-IV)   : {n_strict:,}")
        print(f"  Partially / unclassified          : {n_partial:,}")
        print(f"  Total                             : {len(_wdpa):,}")
        print(f"  Category breakdown:\n{_wdpa['IUCN_CAT'].value_counts().to_string()}")

    pa_distance_data, pa_density_data = compute_pa_layers(
        _wdpa, elev_meta, study_mask
    )
    np.save(_cache_dist,    pa_distance_data)
    np.save(_cache_density, pa_density_data)
    print(f"PA layers cached -> {_cache_dist}, {_cache_density}")

print(f"PA layers ready — shape: {pa_distance_data.shape}")


## 6h — NDVI (30 m → 100 m)
Loads a pre-processed 30 m NDVI composite, `ndvi_germany_2020_30m.tif` (Landsat 8/9 surface reflectance, 2020, exported beforehand from Google Earth Engine — the export script is not part of this notebook), and resamples it bilinearly to the 100 m template. `-9999` sentinels become NaN.


In [ ]:
# ============================================================
# STEP 6h — NDVI from pre-processed 30 m GeoTIFF
# ============================================================

import glob as _glob

NDVI_SRC  = os.path.join(PROJECT_ROOT, "data.nosync/ndvi_germany_2020_30m.tif")
NDVI_OUT  = f"{DATA_DIR}/layers/ndvi_gee_{TARGET_RES}m.tif"

_ndvi_cache = f"{DATA_DIR}/layers/ndvi_{TARGET_RES}m.npy"
if os.path.exists(_ndvi_cache):
    ndvi_data = np.load(_ndvi_cache)
    print(f"NDVI: loaded from cache ({_ndvi_cache})")
else:
    if not os.path.exists(NDVI_SRC):
        raise FileNotFoundError(f"NDVI source not found: {NDVI_SRC}")
    print(f"Resampling NDVI from 30 m → {TARGET_RES} m UTM...")
    ndvi_data = resample_to_template(NDVI_SRC, elev_meta, resampling=Resampling.bilinear)
    ndvi_data[ndvi_data <= -9000] = np.nan   # normalise any -9999 sentinel to NaN
    ndvi_data[study_mask] = np.nan
    np.save(_ndvi_cache, ndvi_data)
    print(f"NDVI: resampled and cached → {_ndvi_cache}")

print(f"NDVI: min={np.nanmin(ndvi_data):.3f}  max={np.nanmax(ndvi_data):.3f}  "
      f"mean={np.nanmean(ndvi_data):.3f}")


## 6i — Summer Precipitation — Warmest Quarter (CHELSA BIO18)

**BIO18** = total precipitation of the warmest quarter, 1981–2010 climatology, stored by CHELSA V2.1 in 0.1 mm (converted to mm for figures only).
Source: [CHELSA v2.1](https://chelsa-climate.org/) COG streamed via `/vsicurl/`.

Summer precipitation is a key driver of vegetation productivity and mast-crop yield,
which directly affects wild boar foraging success. Areas with higher warmest-quarter
rainfall support denser understorey vegetation and larger acorn/beech-mast crops,
attracting higher boar densities.

**Note:** BIO18 uses the *warmest* quarter (≈ June–August in Germany), not a fixed
calendar quarter, so it is spatially consistent across the study area.


In [ ]:
# ============================================================
# STEP 6i — Summer Precipitation — Warmest Quarter (CHELSA BIO18)
# ============================================================
# BIO18 = total precipitation of the warmest quarter.
# STORAGE UNIT: 0.1 mm (CHELSA V2.1, scale 0.1, offset 0) -- the raw values
# here run 1518..11555, i.e. 151.8..1155.5 mm. The GeoTIFF window written below
# does not carry scale/offset metadata, so rasterio reads the raw integers.
# The pipeline keeps them as-is; convert (x * 0.1) only for reporting.
# Streamed from the CHELSA COG via /vsicurl/ — no full download.

def download_chelsa_bio18(gdf, path=os.path.join(DATA_DIR, "map_data")):
    from rasterio.windows import from_bounds as window_from_bounds
    out_path = os.path.join(path, "chelsa_bio18_germany.tif")
    if os.path.exists(out_path):
        print(f"CHELSA BIO18 already present: {out_path}")
        return out_path

    url = ("/vsicurl/https://os.zhdk.cloud.switch.ch/chelsav2/GLOBAL/"
           "climatologies/1981-2010/bio/CHELSA_bio18_1981-2010_V.2.1.tif")

    gdf_wgs84 = gdf.to_crs("EPSG:4326")
    minx, miny, maxx, maxy = gdf_wgs84.total_bounds
    print(f"Reading CHELSA BIO18 window ({minx:.2f},{miny:.2f} \u2013 {maxx:.2f},{maxy:.2f})...")
    with rasterio.open(url) as src:
        win  = window_from_bounds(minx, miny, maxx, maxy, src.transform)
        data = src.read(1, window=win).astype(np.float32)
        tr   = src.window_transform(win)
        profile = src.profile.copy()
        profile.update({"height": data.shape[0], "width": data.shape[1],
                        "transform": tr, "driver": "GTiff",
                        "dtype": "float32", "compress": "lzw", "count": 1})
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(data, 1)
    print(f"Saved CHELSA BIO18 \u2192 {out_path}  ({data.shape[0]}\u00d7{data.shape[1]} px)")
    return out_path

summerPrecip_path = download_chelsa_bio18(regions)

_summerPrecip_cache = f"{DATA_DIR}/layers/summerPrecip_{TARGET_RES}m.npy"
if os.path.exists(_summerPrecip_cache):
    summerPrecip_data = np.load(_summerPrecip_cache)
    print(f"Summer Precipitation: loaded from cache ({_summerPrecip_cache})")
else:
    summerPrecip_data = resample_to_template(summerPrecip_path, elev_meta)
    summerPrecip_data[study_mask] = np.nan
    np.save(_summerPrecip_cache, summerPrecip_data)
    print(f"Summer Precipitation: resampled to {TARGET_RES} m UTM and cached \u2192 {_summerPrecip_cache}")

print(f"\nLocal data base path:", BASE_PATH)
print(f"OSM directories:", OSM_DIRS)


## 8 — Multi-Layer Stack Assembly
Assemble all environmental layers from their `.npy` caches into a single `(H, W, n_layers)` array (14 layers before variable selection). Nothing is written here — the only saved product is the checkpoint in step 14.


In [ ]:
# ============================================================
# STEP 8 — Assemble the Environmental Stack
# ============================================================
# All layers are already cached as individual .npy files.
# We assemble them directly into env_stack_og — no TIFF needed.
#   env_stack_og  shape: (H, W, n_layers)

layer_names_og = [
    "Elev", "Slope", "TRI", "Roughness",
    "WinterTemp", "NDVI", "Population", "Road_Density", "Railway_Density",
    "Landuse", "Water_Distance", "SummerPrecip",
]
arrays_og = [
    elev_data, slope_data, tri_data, roughness_data,
    temp_data, ndvi_data, pop_data, road_data, railway_data, landuse_data, water_data,
    summerPrecip_data,
]

if 'pa_distance_data' in dir() and not np.all(np.isnan(pa_distance_data)):
    layer_names_og.append("PA_Distance")
    arrays_og.append(pa_distance_data)
    print("PA_Distance included in stack.")

if 'pa_density_data' in dir() and not np.all(np.isnan(pa_density_data)):
    layer_names_og.append("PA_Density")
    arrays_og.append(pa_density_data)
    print("PA_Density included in stack.")

env_stack_og = np.stack(arrays_og, axis=-1).astype(np.float32)
del arrays_og
import gc; gc.collect()

study_mask = np.all(np.isnan(env_stack_og), axis=-1)
H, W       = env_stack_og.shape[:2]
print(f"Stack assembled: {env_stack_og.shape}  |  Layers: {layer_names_og}")


## 8a — Resolution Check
Compare the native resolution of each source file against the resampled output in the 100 m stack.

In [ ]:
# Resolution and format summary for all input data sources.
# Reads the actual resolution from each file rather than hardcoding it,
# so the table stays correct even if source files change.

import math, glob as _glob

def get_raster_res(path):
    """Return (res_x_m, res_y_m, width, height, crs_label) for a raster file."""
    try:
        with rasterio.open(path) as src:
            rx = abs(src.transform.a)
            ry = abs(src.transform.e)
            if src.crs and not src.crs.is_projected:
                lat = (src.bounds.top + src.bounds.bottom) / 2
                rx  = rx * 111320 * math.cos(math.radians(lat))
                ry  = ry * 110540
                crs_label = "WGS84 (°→m approx)"
            else:
                crs_label = f"EPSG:{src.crs.to_epsg()}" if src.crs else "unknown"
            return rx, ry, src.width, src.height, crs_label
    except Exception as e:
        return None, None, None, None, f"ERROR: {e}"

osm_files = {
    "Road_Density":     "gis_osm_roads_free_1.shp",
    "Railway_Density":  "gis_osm_railways_free_1.shp",
    "Water_Distance":   ["gis_osm_water_a_free_1.shp",
                         "gis_osm_waterways_free_1.shp"],
}

# ── Print table ────────────────────────────────────────────────────────
HDR = f"{'Layer':<28} {'Source format':<26} {'Native res':>12}  →  {'Output res':>10}  {'Size':>16}"
print(HDR)
print("-" * len(HDR))

# Raster sources
raster_sources = [
    ("Elevation",   "Copernicus DEM GLO-30", os.path.expanduser("~/wolf_sdm_stack/elevation_raw.tif"),
                                             f"{DATA_DIR}/layers/elevation_masked_{TARGET_RES}m.tif"),
    ("Slope",       "derived: Elev 100 m",   f"{DATA_DIR}/layers/elevation_masked_{TARGET_RES}m.tif",
                                             f"{DATA_DIR}/layers/slope_{TARGET_RES}m.npy"),
    ("TRI",         "derived: Elev 100 m",   f"{DATA_DIR}/layers/elevation_masked_{TARGET_RES}m.tif",
                                             f"{DATA_DIR}/layers/tri_{TARGET_RES}m.npy"),
    ("Roughness",   "derived: Elev 100 m",   f"{DATA_DIR}/layers/elevation_masked_{TARGET_RES}m.tif",
                                             f"{DATA_DIR}/layers/roughness_nanaware_{TARGET_RES}m.npy"),
    ("WinterTemp",  "CHELSA BIO11 COG",      f"{DATA_DIR}/map_data/chelsa_bio11_germany.tif",
                                             f"{DATA_DIR}/layers/chelsa_bio11_{TARGET_RES}m.npy"),
    ("SummerPrecip", "CHELSA BIO18 COG",     f"{DATA_DIR}/map_data/chelsa_bio18_germany.tif",
                                             f"{DATA_DIR}/layers/summerPrecip_{TARGET_RES}m.npy"),
    ("NDVI",        "NDVI composite 30 m",   NDVI_SRC,
                                             f"{DATA_DIR}/layers/ndvi_{TARGET_RES}m.npy"),
    ("Population",  "population GeoTIFF",    f"{DATA_DIR}/population_germany_2020.tif",
                                             f"{DATA_DIR}/layers/pop_{TARGET_RES}m.npy"),
    ("Landuse",     "ESA WorldCover 2021",   next(iter(sorted(_glob.glob(os.path.join(
                                                 DATA_DIR, "worldcover_germany_2021_10m-*.tif")))), None),
                                             f"{DATA_DIR}/layers/landuse_score_mode_{TARGET_RES}m.npy"),
]

for layer, fmt, src_path, out_path in raster_sources:
    # Native resolution — read from source file
    if src_path is None or not os.path.exists(src_path):
        src_str  = "not found"
        size_str = ""
    else:
        rx, ry, w, h, _ = get_raster_res(src_path)
        src_str  = f"{rx:>6.0f} m" if rx else "error"
        size_str = f"{w:,} x {h:,} px" if w else ""

    # Output resolution — .npy caches just check existence; rasters read actual res
    if out_path and out_path.endswith(".npy"):
        if os.path.exists(out_path):
            out_str  = f"{TARGET_RES:>6.0f} m"
            size_str = "(.npy cache)"
        else:
            out_str  = "not yet built"
    elif out_path and os.path.exists(out_path):
        rx_out, ry_out, w_out, h_out, _ = get_raster_res(out_path)
        out_str = f"{rx_out:>6.0f} m" if rx_out else "error"
    else:
        out_str = "not yet built"

    print(f"{layer:<28} {fmt:<26} {src_str:>12}  →  {out_str:>10}  {size_str:>16}")

# Vector → raster sources
print()
for layer, filenames in osm_files.items():
    if isinstance(filenames, str):
        filenames = [filenames]
    all_paths = [os.path.join(BASE_PATH, d, fn)
                 for d in OSM_DIRS for fn in filenames]
    found = [p for p in all_paths if os.path.exists(p)]
    n_files = len(found) // len(filenames)
    try:
        import fiona
        total_feats = sum(len(fiona.open(p)) for p in found)
        feat_str = f"{total_feats:,} features"
    except Exception:
        feat_str = f"{n_files}/{len(OSM_DIRS)} dirs"
    fmt = f"OSM shapefile ({n_files}/{len(OSM_DIRS)} dirs)"
    print(f"{layer:<28} {fmt:<26} {'vector':>12}  →  {'rasterized':>10}  {feat_str:>16}")

# ── WDPA / Protected Areas ──────────────────────────────────────────────────
_wdpa_feat = 0
if 'WDPA_SHP_DIR' in dir() and os.path.exists(WDPA_SHP_DIR):
    for _i in range(3):
        _sp = os.path.join(WDPA_SHP_DIR, f"{WDPA_STEM}_{_i}", f"{WDPA_STEM}-polygons.shp")
        if os.path.exists(_sp):
            try:
                import fiona
                _wdpa_feat += len(fiona.open(_sp))
            except Exception:
                pass
_wdpa_cache = os.path.exists(f'{DATA_DIR}/map_data/WDPA_Germany.gpkg')
_wdpa_feat_str = (f"{_wdpa_feat:,} polygons" if _wdpa_feat
                  else ("cached .gpkg" if _wdpa_cache else "not available"))
print(f"{'PA_Distance / PA_Density':<28} {'WDPA WDOECM DEU (3-split shp)':<26} "
      f"{'vector':>12}  →  {'rasterized':>10}  {_wdpa_feat_str:>16}")

# Output stack — read actual resolution from elev_meta (already loaded from file)
print()
rx_out = abs(elev_meta["transform"].a)
ry_out = abs(elev_meta["transform"].e)
print(f"{'-- Final stack (env_stack_og)':<28} "
      f"{'all layers aligned':<26} "
      f"{'various':>12}  →  {rx_out:>9.0f} m  "
      f"{elev_meta['width']:,} x {elev_meta['height']:,} px")
print(f"  (x-res={rx_out:.2f} m, y-res={ry_out:.2f} m — read from elev_meta transform)")


## 9 — Visualization & Export
Plot all layers of the full environmental stack as a multipanel figure.


In [ ]:
# ============================================================
# STEP 9a — Visualise Raw Environmental Layers
# ============================================================
# Before any transformation we plot all layers to visually check
# that each one looks geographically plausible and covers the study area.

# ── Display units: FIGURES ONLY ──────────────────────────────────────────────
# The data path keeps CHELSA's native storage units (BIO11: 0.1 K, BIO18:
# 0.1 mm), so the saved stack is exactly what CHELSA delivers and no conversion
# can drift out of sync with it. Conversion to degC / mm happens only here, on a
# plotting copy. Defined once and reused by every figure cell below.
DISPLAY_UNITS = {
    "WinterTemp":   (lambda v: v * 0.1 - 273.15, "°C"),
    "SummerPrecip": (lambda v: v * 0.1,          "mm"),
}

def to_display(name, arr, logged=False):
    """(values, label) of layer `name` for a FIGURE; the data are never changed.
    logged=True means the layer carries log1p: SummerPrecip is then
    back-transformed to mm, because that is the unit a reader expects; every
    other log1p layer is shown in log1p space and labelled so."""
    if logged and name in DISPLAY_UNITS:
        arr = np.expm1(arr)
        logged = False
    if name in DISPLAY_UNITS:
        f, unit = DISPLAY_UNITS[name]
        return f(arr), f"{name} [{unit}]"
    return arr, (f"{name} (log1p)" if logged else name)

n     = len(layer_names_og)    # number of layers to plot
ncols = 3                       # three panels per row
nrows = -(-n // ncols)          # ceiling division: enough rows to fit all layers

fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 4))
axes = axes.flatten()   # convert 2D array of axes into a 1D list for easy indexing

_step = max(1, max(H, W) // 800)  # downsample for display speed
for i, name in enumerate(layer_names_og):
    ax  = axes[i]
    # imshow displays a 2D array as a colour image; viridis is a perceptually uniform colormap
    _vals, _label = to_display(name, env_stack_og[::_step, ::_step, i])
    im  = ax.imshow(_vals, cmap="viridis", interpolation="nearest")
    ax.set_title(_label, fontsize=10)
    ax.axis("off")    # hide axis ticks and labels — not meaningful for raster maps
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)   # add colour scale bar

# Hide any unused subplot panels (if n is not divisible by ncols)
for j in range(n, len(axes)):
    axes[j].axis("off")

fig.suptitle("Environmental Covariates Stack (100 m UTM)", fontsize=14, fontweight="bold")
plt.tight_layout()   # prevent panels from overlapping
plt.savefig(f"{FIG_DIR}/Environmental_Stack_Multipanel_{TARGET_RES}m.png", dpi=120, bbox_inches="tight")
plt.show()
plt.close(fig)


## 10 — Transformation 


In [ ]:
# ── Is log1p transformation necessary for PA_Distance? ──────────────────────
# Checks skewness, percentiles, and plots raw vs log1p distribution side by side.
from scipy.stats import skew
import matplotlib.pyplot as plt

valid_pa = pa_distance_data[~study_mask & ~np.isnan(pa_distance_data)]

skew_raw  = skew(valid_pa)
skew_log  = skew(np.log1p(valid_pa))
pcts      = np.percentile(valid_pa, [0, 25, 50, 75, 95, 99, 100])

print(f"Pixels inside study area : {len(valid_pa):,}")
print(f"Pixels with PA_Distance=0: {(valid_pa == 0).sum():,}  "
      f"({100*(valid_pa==0).mean():.1f}%  — inside a PA)")
print(f"\nPercentiles (metres):")
for p, v in zip([0, 25, 50, 75, 95, 99, 100], pcts):
    print(f"  p{p:>3}: {v:>10,.0f} m")
print(f"\nSkewness — raw: {skew_raw:.2f}  |  after log1p: {skew_log:.2f}")
print("(rule of thumb: |skew| > 1 → transformation helps)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(valid_pa / 1000, bins=80, color="steelblue", edgecolor="none")
axes[0].set_xlabel("PA Distance (km)")
axes[0].set_title(f"Raw  (skew={skew_raw:.2f})")
axes[1].hist(np.log1p(valid_pa), bins=80, color="darkorange", edgecolor="none")
axes[1].set_xlabel("log1p(PA Distance in m)")
axes[1].set_title(f"log1p  (skew={skew_log:.2f})")
for ax in axes:
    ax.set_ylabel("Pixel count")
plt.suptitle("PA_Distance distribution — raw vs log1p", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 10 — Log Transformation (before collinearity analysis)
# ============================================================
# ORDER MATTERS. log1p is applied HERE, before the collinearity analysis
# and variable selection (section 11), so that Pearson correlations and
# VIFs are computed on the same scale the models later see. This cell does
# ONLY the log transformation. There is no min-max scaling in this notebook;
# the synthetic experiment scales the stack when it loads it.
# Several layers have strongly right-skewed distributions: most pixels
# have low values but a few extreme values dominate the range.
# Example: Population is 0 in forests but >10 000 in city centres.
# A log1p transformation compresses these extremes so the CNN can
# learn from the variation in common (low) values, not just outliers.
#
# log1p(x) = log(1 + x)
#   - log1p(0) = 0  → zero pixels stay at zero (unlike log(x+ε))
#   - only valid for x >= 0  → TPI is excluded (has negative values)
#
# UNITS. The CHELSA layers are used in their native storage units, which
# are integers scaled by 0.1 (BIO18 summer precipitation: 0.1 mm). log1p is
# applied to those native values. At these magnitudes (raw >= 1518)
# log1p(x) and log(x) differ by < 0.07 %, so the result equals log(mm) up to
# an additive constant -- and the min-max scaling in the synthetic experiment removes additive
# constants. Measured: max deviation on the [0,1] scale 0.0006.

# Layers that benefit from log1p transformation (all are non-negative)
# PA_Distance is right-skewed (most pixels far from any PA with outliers near)
# — same treatment as Water_Distance. PA_Density is already [0,1]; no log needed.
LOG_LAYERS = ["Roughness", "TRI", "Population",
              "Railway_Density", "Water_Distance", "PA_Distance", "SummerPrecip"]

# Work on a copy — we want to keep env_stack_og unchanged for reference
env_stack_t = env_stack_og.copy()

for name in LOG_LAYERS:
    if name not in layer_names_og:
        print(f"  [skip] {name} not found in layer_names_og")
        continue

    idx = layer_names_og.index(name)   # find which band index this layer has
    arr = env_stack_t[:, :, idx]       # extract the 2D layer
    valid = arr[~np.isnan(arr)]        # get all non-NaN pixel values

    if (valid < 0).any():
        # log1p is undefined for negative values — skip this layer
        print(f"  [skip] {name} has negative values — log1p not applicable")
        continue

    # Apply log1p; np.where preserves NaN pixels (NaN in → NaN out)
    env_stack_t[:, :, idx] = np.where(np.isnan(arr), np.nan, np.log1p(arr))
    print(f"  log1p: {name:<22} "
          f"range before: [{valid.min():.4f}, {valid.max():.4f}]  "
          f"range after:  [{np.log1p(valid.min()):.4f}, {np.log1p(valid.max()):.4f}]")

# env_stack_og is no longer needed after log-transformation;
# freeing ~2-3 GB of RAM before the collinearity analysis.
import gc
del env_stack_og
gc.collect()
print("env_stack_og freed (~2-3 GB).")


## 11 — Collinearity Analysis & Variable Selection
Compute a **Pearson correlation matrix** on a random sample of valid pixels across all layers.
Highly correlated pairs (|r| > 0.7) are candidates for removal.

After inspecting the heatmap, edit `final_names` to select the final predictor set.
The reduced stack is used for all downstream steps.

The selected set is defined in `final_names` in the selection cell below — that list, not this text, is authoritative. `Landuse` stays in the stack but is not a predictor: it is an ordinal score, tested with Kruskal–Wallis instead of Pearson/VIF, and used by the synthetic experiment only as the source of the refuge field. `Water_Distance` is excluded at selection because the synthetic experiment does not use it.


In [ ]:
# Use env_stack_t (full log-transformed stack, all layers) for correlation analysis.
# Pearson correlation is scale-invariant, so using unscaled values gives identical results.
H, W, C = env_stack_t.shape
flat = env_stack_t.reshape(-1, C)   # flatten H×W to (H*W, C) for sampling

# Sample up to 50 000 pixels (pairwise NaN handled by corr(min_periods=500))
np.random.seed(123)
# Random sample of pixels — Pearson correlation doesn't need the full raster
sample_idx  = np.random.choice(len(flat), size=min(50000, len(flat)), replace=False)
# Exclude Landuse from Pearson correlation (ordinal/categorical — tested separately with Kruskal-Wallis)
_pearson_cols = [n for n in layer_names_og if n != "Landuse"]
sample_data = pd.DataFrame(flat[sample_idx], columns=layer_names_og)
print(f"Sample size: {len(sample_data):,} pixels  |  "
      f"Non-NaN per layer: { {c: int(sample_data[c].notna().sum()) for c in layer_names_og} }")
# min_periods=500: only compute r for pairs that share ≥500 non-NaN rows;
# avoids spurious correlations from sparse layers
cor_matrix  = sample_data[_pearson_cols].corr(min_periods=500)

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.tril(np.ones_like(cor_matrix, dtype=bool), k=-1)   # mask lower triangle to show upper triangle only
sns.heatmap(cor_matrix, mask=mask, annot=True, fmt=".2f", cmap="PuOr",
            center=0, ax=ax, square=True, linewidths=0.5)
# No in-figure title: the thesis carries it in the caption, in the
# document font.
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/Correlation_Heatmap_Initial_{TARGET_RES}m.png", dpi=150, bbox_inches="tight")
plt.show()




In [ ]:
# Kruskal-Wallis test: association between Landuse (ordinal/categorical)
# and each continuous predictor in the final layer set.
#
# Why not Pearson? Pearson assumes both variables are continuous and linearly related.
# Landuse is an ordinal score (0-5 discrete categories) — Pearson underestimates
# its association with continuous variables.
#
# Kruskal-Wallis tests whether the distribution of a continuous variable
# differs significantly across Landuse categories (non-parametric ANOVA).
# A significant result (p < 0.05) means the variable and Landuse are associated.
# Effect size: eta-squared (η²) = H / (n - 1), ranges 0–1.

from scipy.stats import kruskal

# Work on the full pre-selection sample (sample_data from collinearity cell)
landuse_col = "Landuse"
continuous  = [n for n in layer_names_og if n != landuse_col]

# Drop NaN rows
s = sample_data.dropna()
landuse_cats = sorted(s[landuse_col].unique())

print(f"Kruskal-Wallis test: Landuse ({len(landuse_cats)} categories) vs all layers (pre-selection)")
print(f"Sample size: {len(s):,} pixels")
print(f"Landuse categories (score): {[round(c,3) for c in landuse_cats]}")
print()
print(f"  {'Variable':<20} {'H-stat':>8} {'p-value':>12} {'eta2':>8}  {'Sig':>5}")
print("  " + "-" * 60)

kw_results = {}
for var in continuous:
    groups = [s.loc[s[landuse_col] == cat, var].values for cat in landuse_cats]
    groups = [g for g in groups if len(g) >= 5]   # kruskal() requires at least 5 samples per group
    if len(groups) < 2:
        print(f"  {var:<20} {'—':>8} {'not enough data':>12}")
        continue
    try:
        H, p = kruskal(*groups)
    except ValueError:
        # All values identical in at least one group (e.g. Population = 0 everywhere)
        print(f"  {var:<20} {'—':>8} {'all values identical':>20}")
        continue
    n    = sum(len(g) for g in groups)
    eta2 = (H - len(groups) + 1) / (n - len(groups))
    eta2 = max(eta2, 0.0)
    sig  = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "n.s."))
    kw_results[var] = {"H": H, "p": p, "eta2": eta2}
    print(f"  {var:<20} {H:>8.1f} {p:>12.2e} {eta2:>8.4f}  {sig:>5}")

print()
print("Interpretation:")
print("  eta2 < 0.01 : negligible association")
print("  eta2 < 0.06 : small association")
print("  eta2 < 0.14 : medium association")
print("  eta2 >= 0.14: large association")

# Bar plot of eta2
fig, ax = plt.subplots(figsize=(6, 3.5))
vars_sorted = sorted(kw_results, key=lambda v: kw_results[v]["eta2"], reverse=True)
eta2_vals   = [kw_results[v]["eta2"] for v in vars_sorted]
bars = ax.barh(vars_sorted, eta2_vals, color="#4a90d9", alpha=0.85)
ax.set_xlim(0, 0.40)
ax.set_xlabel("Effect size (eta squared)")
ax.set_title("Kruskal-Wallis: Association with Landuse\n(non-parametric, ordinal-safe)")
for bar, val in zip(bars, eta2_vals):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f"{val:.4f}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/Landuse_KruskalWallis_{TARGET_RES}m.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Variance Inflation Factor — ALL layers (pre-selection) ──────────────────
# VIF quantifies how much the variance of each regression coefficient is
# inflated by multicollinearity with the other predictors.
# VIF = 1 / (1 - R²_i), where R²_i is the R² of regressing variable i on all others.
#
# Thresholds (common convention):
#   VIF < 5   → acceptable
#   VIF 5–10  → moderate collinearity, investigate
#   VIF > 10  → high collinearity, consider removing
#
# Run this cell BEFORE variable selection to inform which variables to drop.
# Post-selection VIF is computed in the cell immediately after variable selection.

from statsmodels.stats.outliers_influence import variance_inflation_factor

CATEGORICAL_LAYERS = ["Landuse"]  # excluded from VIF (ordinal codes ≠ continuous)

def compute_vif(df):
    """Return a DataFrame with VIF, max|r|, and partner for each column.

    An explicit constant column is added internally — statsmodels does NOT
    add one automatically, and omitting it forces auxiliary regressions
    through the origin, inflating VIF for any variable with a non-zero mean
    (every predictor in this stack qualifies).  Landuse must be excluded before
    calling this function.
    """
    clean = df.dropna().reset_index(drop=True)
    col_names = list(clean.columns)

    # Add constant BEFORE passing to variance_inflation_factor
    X = clean.copy()
    X["const"] = 1.0
    X_vals = X.values

    vif_vals = [
        variance_inflation_factor(X_vals, i)
        for i in range(len(col_names))   # indices 0..n-1 = real predictors; n = const
    ]

    # Pairwise |r| matrix for self-consistency check and reporting
    cor = clean.corr().abs()
    np.fill_diagonal(cor.values, 0.0)
    max_r       = cor.max()
    max_partner = cor.idxmax()

    result = pd.DataFrame({
        "Variable": col_names,
        "VIF":      vif_vals,
        "max|r|":   [max_r[v]       for v in col_names],
        "partner":  [max_partner[v] for v in col_names],
    }).sort_values("VIF", ascending=False).reset_index(drop=True)

    # Self-consistency assertion:
    # VIF >= 1/(1-r_max^2) by construction (pairwise r is a lower bound).
    # A VIF that is both > 10 AND has max|r| < 0.7 cannot arise from real
    # multicollinearity — it signals a computation problem.
    for _, row in result.iterrows():
        lb = 1.0 / (1.0 - row["max|r|"] ** 2 + 1e-10)
        if row["VIF"] > 10 and row["max|r|"] < 0.70:
            warnings.warn(
                f"VIF SANITY FAILED: '{row.Variable}' VIF={row.VIF:.2f} but "
                f"max|r|={row['max|r|']:.2f} (partner={row.partner}). "
                "VIF > 10 with max|r| < 0.70 is not mathematically possible — "
                "check for a missing intercept or categorical column in the design matrix.",
                RuntimeWarning, stacklevel=3,
            )

    return result

vif_all = compute_vif(sample_data[layer_names_og].drop(columns=CATEGORICAL_LAYERS))

def _bar_color(vif):
    if vif > 10: return "#D55E00"
    if vif > 5:  return "#E69F00"
    return              "#009E73"

print("VIF — all layers (pre-selection, Landuse excluded as categorical):")
print(f"  {'Variable':<20} {'VIF':>7}  {'max|r|':>7}  {'partner':<20}  Flag")
print("  " + "-" * 70)
for _, row in vif_all.iterrows():
    flag = "HIGH" if row.VIF > 10 else ("MODERATE" if row.VIF > 5 else "ok")
    print(f"  {row.Variable:<20} {row.VIF:>7.2f}  {row['max|r|']:>7.2f}  {row.partner:<20}  {flag}")
if vif_all["max|r|"].max() >= 0.70:
    pairs = vif_all[vif_all["max|r|"] >= 0.70][["Variable", "max|r|", "partner"]]
    print()
    print("  ⚠ |r| ≥ 0.70 — flag for layer-selection experiment (do not auto-drop):")
    for _, p in pairs.iterrows():
        print(f"    {p.Variable} — {p.partner}  r={p['max|r|']:.2f}")

fig, ax = plt.subplots(figsize=(7, 5))
colors = [_bar_color(v) for v in vif_all.VIF]
bars   = ax.barh(vif_all.Variable[::-1], vif_all.VIF[::-1],
                 color=colors[::-1], edgecolor="white", height=0.6)
ax.axvline(5,  color="#E69F00", linestyle="--", linewidth=1.2, label="VIF = 5")
ax.axvline(10, color="#D55E00", linestyle="--", linewidth=1.2, label="VIF = 10")
ax.set_xlabel("Variance Inflation Factor (VIF)")
# No in-figure title: the thesis carries it in the caption, in the
# document font.
ax.legend(fontsize=8)
for bar, val in zip(bars, vif_all.VIF[::-1]):
    ax.text(val + 0.1, bar.get_y() + bar.get_height() / 2,
            f"{val:.2f}", va="center", fontsize=8)
ax.set_xlim(0, max(vif_all.VIF.max(), 12) * 1.15)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/VIF_Analysis_PreSelection_{TARGET_RES}m.png", dpi=150, bbox_inches="tight")
plt.show()


# Selection of final layers

In [ ]:
# Edit final_names based on the KW and VIF results above.
# Available: "Elev", "Slope", "TRI", "Roughness", "WinterTemp", "SummerPrecip",
#            "Population", "Road_Density", "Railway_Density",
#            "Landuse", "Water_Distance", "NDVI"
# + (if WDPA data available): "PA_Distance", "PA_Density"
#
# Exclusion criteria (adjust based on current KW + VIF output):
#   |r| > 0.7  → drop the less ecologically relevant of the pair
#   VIF > 10   → strong signal to remove
#   KW eta2 < 0.01 → negligible association with Landuse (weak predictor)
#
# WATER_DISTANCE IS EXCLUDED HERE, not downstream. The synthetic experiment never
# uses it, so carrying it into the saved stack only to drop it on load would put
# a layer into the collinearity/VIF results and the checkpoint that no analysis
# reads. Excluding it at selection keeps the reported predictor set identical to
# the one the experiment actually uses.
#
# LANDUSE STAYS IN THE STACK, but not as a predictor: the synthetic notebook takes
# it out on load as the raw habitat-quality score (the source of the real-land-use
# refuge field W) and removes it from the model inputs. It is kept here because
# that notebook needs it, and it is analysed here separately (Kruskal-Wallis),
# never through Pearson or VIF.
#
# ELEV IS EXCLUDED (collinearity). With the real BIO11, Elev correlates with
# WinterTemp at r = -0.82 and with SummerPrecip at r = +0.72, and its VIF is 6.0 --
# above both thresholds stated above. Elevation is a DISTAL variable: it acts on a
# species only through temperature, precipitation and terrain, and all three stay
# in the set (WinterTemp, SummerPrecip, Roughness). Keeping the proximal variables
# and dropping the distal proxy is the standard resolution (Austin 2002; Dormann
# et al. 2013). Measured without Elev: max |r| = 0.52, every VIF < 1.7.
# Elevation remains the grid template and the source of Roughness.
final_names = ["Roughness", "Landuse",
               "NDVI", "Road_Density", "PA_Distance",
               "SummerPrecip", "WinterTemp"]

# Select layers from the log-transformed stack (env_stack_t).
final_idx = [layer_names_og.index(n) for n in final_names]
env_final = env_stack_t[:, :, final_idx]

# Sample for correlation matrix and VIF — correlations are scale-invariant
# so using the log-transformed, unscaled values is correct here.
flat2 = env_final.reshape(-1, len(final_names))
np.random.seed(123)
idx2    = np.random.choice(len(flat2), size=min(50000, len(flat2)), replace=False)
sample2 = pd.DataFrame(flat2[idx2], columns=final_names)
cor_matrix_new = sample2.corr()

fig, ax = plt.subplots(figsize=(8, 7))
mask2 = np.tril(np.ones_like(cor_matrix_new, dtype=bool), k=-1)
sns.heatmap(cor_matrix_new, mask=mask2, annot=True, fmt=".2f", cmap="PuOr",
            center=0, ax=ax, square=True, linewidths=0.5)
# No in-figure title: the thesis carries it in the caption, in the
# document font.
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/Correlation_Heatmap_Final_{TARGET_RES}m.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Final layers selected ({len(final_names)}): {final_names}")

### VIF Validation — Selected Variables
Confirms that multicollinearity is resolved after variable selection. All VIF should be < 5.

In [ ]:
# ── VIF on selected variables (post-selection validation) ────────────────────
# Verifies that removing collinear variables actually resolved multicollinearity.
# All remaining variables should have VIF < 5.

vif_final = compute_vif(sample2.drop(columns=["Landuse"], errors="ignore"))

print("VIF — selected variables (post-selection):")
print(f"  {'Variable':<20} {'VIF':>7}  {'max|r|':>7}  {'partner':<20}  Flag")
print("  " + "-" * 70)
all_ok = True
for _, row in vif_final.iterrows():
    flag = "HIGH" if row.VIF > 10 else ("MODERATE" if row.VIF > 5 else "ok")
    if row.VIF > 5: all_ok = False
    print(f"  {row.Variable:<20} {row.VIF:>7.2f}  {row['max|r|']:>7.2f}  {row.partner:<20}  {flag}")
print()
print("✓ All VIF < 5 — collinearity resolved." if all_ok else
      "⚠ Some VIF still elevated — consider further variable removal.")
if vif_final["max|r|"].max() >= 0.70:
    pairs = vif_final[vif_final["max|r|"] >= 0.70][["Variable", "max|r|", "partner"]]
    print()
    print("  ⚠ |r| ≥ 0.70 — flag for layer-selection experiment:")
    for _, p in pairs.iterrows():
        print(f"    {p.Variable} — {p.partner}  r={p['max|r|']:.2f}")

fig, ax = plt.subplots(figsize=(6, max(3, len(vif_final) * 0.5 + 1)))
colors = [_bar_color(v) for v in vif_final.VIF]
bars   = ax.barh(vif_final.Variable[::-1], vif_final.VIF[::-1],
                 color=colors[::-1], edgecolor="white", height=0.6)
ax.axvline(5,  color="#E69F00", linestyle="--", linewidth=1.2, label="VIF = 5")
ax.axvline(10, color="#D55E00", linestyle="--", linewidth=1.2, label="VIF = 10")
ax.set_xlabel("Variance Inflation Factor (VIF)")
# No in-figure title: the thesis carries it in the caption, in the
# document font.
ax.legend(fontsize=8)
for bar, val in zip(bars, vif_final.VIF[::-1]):
    ax.text(val + 0.05, bar.get_y() + bar.get_height() / 2,
            f"{val:.2f}", va="center", fontsize=8)
ax.set_xlim(0, max(vif_final.VIF.max(), 6) * 1.2)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/VIF_Analysis_PostSelection_{TARGET_RES}m.png", dpi=150, bbox_inches="tight")
plt.show()


## 12 — Distributions of the Final Predictors
One histogram per selected predictor, on the scale that is saved to the checkpoint (log1p where applied, otherwise native units). WinterTemp and SummerPrecip are shown in °C and mm — **for display only**. This notebook does **not** scale; the synthetic experiment scales the stack when it loads it.

In [ ]:
# ============================================================
# STEP 12 — Distribution of the final predictors
# ============================================================
# One panel per selected predictor, on the scale saved to the checkpoint:
# log1p-transformed where applied, otherwise native units. WinterTemp and
# SummerPrecip are converted to degC / mm for display only (to_display, defined
# in the raw-stack figure cell). This replaces a single overlay of min-max-scaled
# values, which only made sense while this notebook scaled -- it no longer does.
_flat = env_final.reshape(-1, len(final_names))
np.random.seed(123)
_idx  = np.random.choice(len(_flat), size=min(50000, len(_flat)), replace=False)
_smp  = pd.DataFrame(_flat[_idx], columns=final_names).dropna()

ncols = 3
nrows = -(-len(final_names) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3.2))
axes = axes.flatten()
for i, name in enumerate(final_names):
    _vals, _label = to_display(name, _smp[name].values, logged=name in LOG_LAYERS)
    axes[i].hist(_vals, bins=60, color="#4a90d9", alpha=0.85)
    axes[i].set_xlabel(_label)
    axes[i].set_ylabel("Pixels")
for j in range(len(final_names), len(axes)):
    axes[j].axis("off")
fig.suptitle(f"Final predictors — distributions ({len(_smp):,} sampled pixels)",
             fontweight="bold")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/Final_Predictor_Distributions_{TARGET_RES}m.png", dpi=300, bbox_inches="tight")
plt.show()

## 13 — Bridge: Final Stack and Metadata
Exposes the selected, log-transformed, **unscaled** stack as `env_stack_final`, and defines `raster_transform`, `raster_crs`, `LAYER_NAMES` and `N_LAYERS` for the checkpoint.

In [ ]:
# ============================================================
# STEP 13 — Assign Working Variables
# ============================================================
# After variable selection, env_final holds the selected layers, log1p-transformed
# where applicable and NOT scaled. It is exposed as env_stack_final -- the name no
# longer says "scaled", because nothing in this notebook scales any more; the
# synthetic experiment does that when it loads the stack.
env_stack_final  = env_final                  # (H, W, N_LAYERS), log1p where applied, unscaled
raster_transform = elev_meta["transform"]     # affine transform: pixel <-> coordinates
raster_crs       = elev_meta["crs"]           # UTM 32N
H, W, N_LAYERS   = env_stack_final.shape
LAYER_NAMES      = list(final_names)

print(f"Stack shape : {H} × {W} px, {N_LAYERS} layers")
print(f"CRS         : {raster_crs}")
print(f"Layers      : {LAYER_NAMES}")

In [ ]:
# ============================================================
# STEP 13b — Visualise the final predictors (as saved)
# ============================================================
# The exact layers written to the checkpoint. log1p layers are shown in log1p
# space, except SummerPrecip, which is back-transformed to mm; WinterTemp is shown
# in degC. Both conversions are display-only (to_display). There is no fixed
# [0, 1] colour range any more -- the stack is unscaled -- so each panel uses its
# own 2nd-98th percentile range.
n     = len(LAYER_NAMES)
ncols = 3
nrows = -(-n // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 4))
axes = axes.flatten()

_step = max(1, max(H, W) // 800)  # downsample for display speed
for i, name in enumerate(LAYER_NAMES):
    ax = axes[i]
    _vals, _label = to_display(name, env_stack_final[::_step, ::_step, i],
                               logged=name in LOG_LAYERS)
    _v = _vals[np.isfinite(_vals)]
    im = ax.imshow(_vals, cmap="viridis", interpolation="nearest",
                   vmin=np.percentile(_v, 2), vmax=np.percentile(_v, 98))
    ax.set_title(_label, fontsize=12)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
for j in range(n, len(axes)):
    axes[j].axis("off")

fig.suptitle("Final predictors — as saved (log1p where applied; colour range p2–p98)",
             fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/Environmental_Stack_Final_{TARGET_RES}m.png", dpi=120, bbox_inches="tight")
plt.show()
plt.close(fig)

## Checkpoint — Save the Covariate Stack for the Synthetic Experiment
Writes only what this notebook produces: the selected, log-transformed, unscaled covariate stack and the metadata `synthetic_experiment.ipynb.nosync Kopie.ipynb` needs. No species data, splits or model inputs.

In [ ]:
# ============================================================
# STEP 14 — Save the covariate stack for the synthetic experiment
# ============================================================
# WHAT IS SAVED, AND WHY NOTHING ELSE.
# This notebook's product is the selected covariate stack after log1p. Nothing
# else: no scaling (the synthetic experiment scales on load), no species data, no
# splits, no model inputs -- those pipelines are not part of this notebook.
#
# FILE NAME: env_stack_log.npy, not env_scaled.npy. The stack is not scaled, and
# a file called "scaled" that is not would be read wrongly by the next person.
#
# FOLDER: a NEW folder, synthetic_stack_<res>m. The old checkpoint_<res>m still
# holds the inputs of the v5 run and of the real-data model; writing into it would
# silently change what those read.
import json as _json, datetime as _dt
from pathlib import Path

CKPT = Path(CODE_OUT) / SPECIES / f"synthetic_stack_{TARGET_RES}m"
CKPT.mkdir(parents=True, exist_ok=True)
STACK_FILE = "env_stack_log.npy"

# ── Checks before anything is written ────────────────────────────────────────
_st = env_stack_final.astype(np.float32)
assert np.isfinite(_st).any(axis=2).any(), "stack is entirely NaN"
assert not np.isinf(_st).any(), "stack contains +/-inf"
_lu = _st[:, :, LAYER_NAMES.index("Landuse")]
assert set(np.unique(_lu[np.isfinite(_lu)]).astype(int)) <= {0, 1, 2, 3, 4, 5}, \
    "Landuse must be the raw habitat-quality score 0-5"
_t = float(np.nanmedian(_st[:, :, LAYER_NAMES.index("WinterTemp")])) * 0.1 - 273.15
assert _t < 5.0, f"WinterTemp median {_t:.2f} degC -- annual mean, not BIO11"
assert np.nanmin(_st[:, :, LAYER_NAMES.index("WinterTemp")]) > 100.0, \
    "WinterTemp looks min-max scaled -- this checkpoint must be UNscaled (raw BIO11 is ~2600-2800, 0.1 K)"

np.save(CKPT / STACK_FILE, _st)

meta = {
    "STACK_FILE":       STACK_FILE,
    "LAYER_NAMES":      LAYER_NAMES,
    "N_LAYERS":         int(N_LAYERS),
    "H":                int(H),
    "W":                int(W),
    "TARGET_RES":       int(TARGET_RES),
    "raster_crs":       str(raster_crs),
    "raster_transform": list(raster_transform)[:6],
    "nodata":           "NaN",
    "scaled":           False,
    "log1p_layers":     [n for n in LAYER_NAMES if n in LOG_LAYERS],
    "native_units": {
        "WinterTemp":   "CHELSA V2.1 BIO11, 0.1 K (degC = x*0.1 - 273.15)",
        "SummerPrecip": "CHELSA V2.1 BIO18, 0.1 mm (mm = x*0.1), stored AFTER log1p",
        "Landuse":      "habitat-quality score 0-5 (integer-valued)",
    },
    "created":          _dt.datetime.now().isoformat(timespec="seconds"),
}
with open(CKPT / "meta.json", "w") as fh:
    _json.dump(meta, fh, indent=2)

print(f"Saved to {CKPT}/")
print(f"  {STACK_FILE} : {_st.shape}  float32, log1p on {meta['log1p_layers']}, unscaled")
print(f"  meta.json    : {list(meta)}")
del _st, _lu